In [14]:
#@title Cell 24.1 - Notebook overview
from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 24: Targeted AMR-Sequence Alignment and Kernel Construction

## Purpose

Notebook 24 will convert the targeted nucleotide sequences extracted in
Notebook 23 into two sequence-similarity kernel candidates for Model C.

It will:

1. locate and validate all 906 Notebook 23 batch archives;
2. confirm that the 9,058 Model C pathogens are represented once;
3. combine the extracted sequences and sequence-status records;
4. determine which of the 297 planned loci have accepted pathogen sequences;
5. select one sequence for each pathogen and represented locus;
6. define one alignment reference sequence for each represented locus;
7. perform a multiple-sequence alignment separately within each locus;
8. represent each aligned nucleotide as A, C, G, T or an alignment gap;
9. distinguish an alignment gap from a missing or unreliable sequence result;
10. validate the aligned sequences and their aligned sequence lengths;
11. calculate a base-weighted kernel, in which every aligned nucleotide base
    has equal weight;
12. calculate a locus-weighted kernel, in which every represented locus has
    equal weight; and
13. validate, compare and save both sequence-kernel candidates:

\[
K_{\mathrm{seq,base}}
\in\mathbb{R}^{9058\times9058},
\qquad
K_{\mathrm{seq,locus}}
\in\mathbb{R}^{9058\times9058}.
\]

Both kernels will be retained for predictive comparison in Notebook 25.

## Model boundary

Notebook 23 files will be treated as read-only inputs. The existing Model 3B
notebooks and their 9,377-pathogen results will not be changed.

Notebook 24 will use only the 9,058 Model C pathogens with assembly accessions.
Of the 297 planned loci, 265 have accepted pathogen sequences and are included
in the sequence kernels. The remaining 32 loci are retained in the audit but
do not contribute to either sequence kernel.

A missing or unreliable sequence result will not be interpreted as a biological
alignment gap.

Whole-genome similarity, integration with the Model 3B pathogen kernel,
selection between the two sequence kernels and MIC model fitting are outside
the scope of this notebook.

## Expected notebook length

Notebook 24 contains **13 cells**, including Cell 24.10B.
"""))

print(
    "Transition: The next cell will import the required packages "
    "and define the Notebook 24 settings."
)


# Notebook 24: Targeted AMR-Sequence Alignment and Kernel Construction

## Purpose

Notebook 24 will convert the targeted nucleotide sequences extracted in
Notebook 23 into two sequence-similarity kernel candidates for Model C.

It will:

1. locate and validate all 906 Notebook 23 batch archives;
2. confirm that the 9,058 Model C pathogens are represented once;
3. combine the extracted sequences and sequence-status records;
4. determine which of the 297 planned loci have accepted pathogen sequences;
5. select one sequence for each pathogen and represented locus;
6. define one alignment reference sequence for each represented locus;
7. perform a multiple-sequence alignment separately within each locus;
8. represent each aligned nucleotide as A, C, G, T or an alignment gap;
9. distinguish an alignment gap from a missing or unreliable sequence result;
10. validate the aligned sequences and their aligned sequence lengths;
11. calculate a base-weighted kernel, in which every aligned nucleotide base
    has equal weight;
12. calculate a locus-weighted kernel, in which every represented locus has
    equal weight; and
13. validate, compare and save both sequence-kernel candidates:

\[
K_{\mathrm{seq,base}}
\in\mathbb{R}^{9058\times9058},
\qquad
K_{\mathrm{seq,locus}}
\in\mathbb{R}^{9058\times9058}.
\]

Both kernels will be retained for predictive comparison in Notebook 25.

## Model boundary

Notebook 23 files will be treated as read-only inputs. The existing Model 3B
notebooks and their 9,377-pathogen results will not be changed.

Notebook 24 will use only the 9,058 Model C pathogens with assembly accessions.
Of the 297 planned loci, 265 have accepted pathogen sequences and are included
in the sequence kernels. The remaining 32 loci are retained in the audit but
do not contribute to either sequence kernel.

A missing or unreliable sequence result will not be interpreted as a biological
alignment gap.

Whole-genome similarity, integration with the Model 3B pathogen kernel,
selection between the two sequence kernels and MIC model fitting are outside
the scope of this notebook.

## Expected notebook length

Notebook 24 contains **13 cells**, including Cell 24.10B.


Transition: The next cell will import the required packages and define the Notebook 24 settings.


In [2]:
#@title Cell 24.2 - Import packages and define settings
# This cell imports the required packages, connects Google Drive and defines
# the fixed dimensions, nucleotide states, input paths and output directories.

from collections import Counter, defaultdict
from pathlib import Path
import hashlib
import importlib.util
import io
import json
import re
import shutil
import subprocess
import sys
import zipfile

if importlib.util.find_spec("Bio") is None:
    print("Installing Biopython...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "biopython",
        ],
        check=True,
    )

import numpy as np
import pandas as pd
from Bio import AlignIO, SeqIO
from IPython.display import display
from google.colab import drive


drive.mount("/content/drive")

# ---------------------------------------------------------------------------
# Fixed dimensions
# ---------------------------------------------------------------------------

EXPECTED_MODEL_C_PATHOGENS = 9058
EXPECTED_LOCI = 297
EXPECTED_BATCHES = 906
ASSEMBLIES_PER_BATCH = 10

NUCLEOTIDE_STATES = (
    "A",
    "C",
    "G",
    "T",
    "-",
)

assert len(NUCLEOTIDE_STATES) == 5

# ---------------------------------------------------------------------------
# Project directories
# ---------------------------------------------------------------------------

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/Model3_MIC_Project"
)

NOTEBOOK23_DIRECTORY = (
    PROJECT_DIRECTORY / "notebook23"
)

NOTEBOOK23_FULL_EXTRACTION_DIRECTORY = (
    NOTEBOOK23_DIRECTORY / "full_extraction"
)

NOTEBOOK23_BATCH_DIRECTORY = (
    NOTEBOOK23_FULL_EXTRACTION_DIRECTORY / "batches"
)

NOTEBOOK24_DIRECTORY = (
    PROJECT_DIRECTORY / "notebook24"
)

NOTEBOOK24_CHECKPOINT_DIRECTORY = (
    NOTEBOOK24_DIRECTORY / "alignment_checkpoints"
)

NOTEBOOK24_RESULT_DIRECTORY = (
    NOTEBOOK24_DIRECTORY / "results"
)

for directory in [
    NOTEBOOK24_DIRECTORY,
    NOTEBOOK24_CHECKPOINT_DIRECTORY,
    NOTEBOOK24_RESULT_DIRECTORY,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# ---------------------------------------------------------------------------
# Notebook 23 input files
# ---------------------------------------------------------------------------

PROGRESS_PATH = (
    NOTEBOOK23_FULL_EXTRACTION_DIRECTORY
    / "23_full_extraction_progress.csv"
)

LOCUS_PANEL_PATH = (
    NOTEBOOK23_DIRECTORY
    / "23_target_amr_locus_panel.csv"
)

REFERENCE_AUDIT_PATH = (
    NOTEBOOK23_DIRECTORY
    / "23_target_amr_reference_audit.csv"
)

REFERENCE_FASTA_PATH = (
    NOTEBOOK23_DIRECTORY
    / "23_target_amr_reference_sequences.fasta"
)

BATCH_FILENAME_PATTERN = re.compile(
    r"23_batch_(\d{4})\.zip$"
)

settings_summary = pd.DataFrame(
    [
        {
            "setting": "Expected Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "setting": "Expected targeted loci",
            "value": EXPECTED_LOCI,
        },
        {
            "setting": "Expected NB23 batches",
            "value": EXPECTED_BATCHES,
        },
        {
            "setting": "Assemblies per NB23 batch",
            "value": ASSEMBLIES_PER_BATCH,
        },
        {
            "setting": "Aligned nucleotide states",
            "value": ", ".join(NUCLEOTIDE_STATES),
        },
        {
            "setting": "Notebook 24 output directory",
            "value": str(NOTEBOOK24_DIRECTORY),
        },
    ]
)

display(settings_summary)

print(
    "Notebook 24 packages, directories and fixed settings "
    "were defined successfully."
)

print(
    "\nTransition: Cell 24.3 will locate and validate the "
    "available Notebook 23 batch archives without modifying them."
)

Installing Biopython...
Mounted at /content/drive


,setting,value
0,Expected Model C pathogens,9058
1,Expected targeted loci,297
2,Expected NB23 batches,906
3,Assemblies per NB23 batch,10
4,Aligned nucleotide states,"A, C, G, T, -"
5,Notebook 24 output directory,/content/drive/MyDrive/Model3_MIC_Project/note...


Notebook 24 packages, directories and fixed settings were defined successfully.

Transition: Cell 24.3 will locate and validate the available Notebook 23 batch archives without modifying them.


In [3]:
#@title Cell 24.3 - Validate Notebook 23 batch archives
# This cell locates every completed Notebook 23 batch archive and confirms
# that its files, pathogen accessions, batch order and validation status are correct.

required_input_paths = [
    PROGRESS_PATH,
    LOCUS_PANEL_PATH,
    REFERENCE_AUDIT_PATH,
    REFERENCE_FASTA_PATH,
    NOTEBOOK23_BATCH_DIRECTORY,
]

missing_input_paths = [
    path
    for path in required_input_paths
    if not path.exists()
]

if missing_input_paths:
    raise FileNotFoundError(
        f"Required Notebook 23 inputs are missing: "
        f"{missing_input_paths}"
    )

# ---------------------------------------------------------------------------
# Validate the canonical Model C pathogen order
# ---------------------------------------------------------------------------

progress = pd.read_csv(
    PROGRESS_PATH,
    dtype={
        "biosample": str,
        "assembly_accession": str,
        "batch_archive": str,
    },
)

required_progress_columns = {
    "model_c_row_index",
    "model_3b_row_index",
    "biosample",
    "assembly_accession",
    "batch_id",
    "processing_status",
}

missing_progress_columns = (
    required_progress_columns
    - set(progress.columns)
)

if missing_progress_columns:
    raise ValueError(
        "The Notebook 23 progress file is missing columns: "
        f"{sorted(missing_progress_columns)}"
    )

progress["model_c_row_index"] = pd.to_numeric(
    progress["model_c_row_index"],
    errors="raise",
).astype(int)

progress["batch_id"] = pd.to_numeric(
    progress["batch_id"],
    errors="raise",
).astype(int)

progress = progress.sort_values(
    "model_c_row_index"
).reset_index(drop=True)

if len(progress) != EXPECTED_MODEL_C_PATHOGENS:
    raise ValueError(
        f"Expected {EXPECTED_MODEL_C_PATHOGENS:,} pathogens, "
        f"but the progress file contains {len(progress):,}."
    )

expected_row_indices = np.arange(
    EXPECTED_MODEL_C_PATHOGENS
)

if not np.array_equal(
    progress["model_c_row_index"].to_numpy(),
    expected_row_indices,
):
    raise ValueError(
        "The Model C row indices are incomplete or out of order."
    )

if progress["biosample"].duplicated().any():
    raise ValueError(
        "Duplicate BioSamples were found in the progress file."
    )

if progress["assembly_accession"].duplicated().any():
    raise ValueError(
        "Duplicate assembly accessions were found in the "
        "progress file."
    )

expected_batch_ids = (
    progress["model_c_row_index"]
    // ASSEMBLIES_PER_BATCH
    + 1
)

if not np.array_equal(
    progress["batch_id"].to_numpy(),
    expected_batch_ids.to_numpy(),
):
    raise ValueError(
        "The batch identifiers do not match the Model C "
        "pathogen order."
    )

# ---------------------------------------------------------------------------
# Locate the completed batch archives
# ---------------------------------------------------------------------------

batch_archives = []

for archive_path in sorted(
    NOTEBOOK23_BATCH_DIRECTORY.glob(
        "23_batch_*.zip"
    )
):
    match = BATCH_FILENAME_PATTERN.fullmatch(
        archive_path.name
    )

    if match is None:
        raise ValueError(
            f"Unexpected batch filename: {archive_path.name}"
        )

    batch_archives.append(
        (
            int(match.group(1)),
            archive_path,
        )
    )

if not batch_archives:
    raise FileNotFoundError(
        "No completed Notebook 23 batch archives were found."
    )

available_batch_ids = [
    batch_id
    for batch_id, _ in batch_archives
]

if len(available_batch_ids) != len(
    set(available_batch_ids)
):
    raise ValueError(
        "Duplicate Notebook 23 batch identifiers were found."
    )

invalid_batch_ids = [
    batch_id
    for batch_id in available_batch_ids
    if batch_id < 1 or batch_id > EXPECTED_BATCHES
]

if invalid_batch_ids:
    raise ValueError(
        f"Invalid batch identifiers were found: "
        f"{invalid_batch_ids}"
    )

# ---------------------------------------------------------------------------
# Validate every available archive
# ---------------------------------------------------------------------------

required_archive_members = {
    "batch_manifest.csv",
    "target_sequences.csv",
    "amrfinder_results.csv",
    "coding_blast_results.csv",
    "review_items.csv",
    "target_sequences.fasta",
    "batch_summary.json",
}

archive_audit_rows = []
represented_accessions = set()

for batch_id, archive_path in batch_archives:
    with zipfile.ZipFile(
        archive_path,
        "r",
    ) as archive:
        damaged_member = archive.testzip()

        if damaged_member is not None:
            raise ValueError(
                f"{archive_path.name} contains a damaged file: "
                f"{damaged_member}"
            )

        archive_members = set(archive.namelist())

        missing_archive_members = (
            required_archive_members
            - archive_members
        )

        if missing_archive_members:
            raise ValueError(
                f"{archive_path.name} is missing files: "
                f"{sorted(missing_archive_members)}"
            )

        with archive.open(
            "batch_manifest.csv"
        ) as file_handle:
            batch_manifest = pd.read_csv(
                file_handle,
                dtype=str,
            )

        with archive.open(
            "batch_summary.json"
        ) as file_handle:
            batch_summary = json.load(
                file_handle
            )

    required_manifest_columns = {
        "biosample",
        "assembly_accession",
        "processing_status",
    }

    missing_manifest_columns = (
        required_manifest_columns
        - set(batch_manifest.columns)
    )

    if missing_manifest_columns:
        raise ValueError(
            f"{archive_path.name} has an incomplete manifest: "
            f"{sorted(missing_manifest_columns)}"
        )

    if (
        batch_manifest["processing_status"]
        != "completed"
    ).any():
        raise ValueError(
            f"{archive_path.name} contains an incomplete "
            "assembly result."
        )

    expected_batch_accessions = (
        progress.loc[
            progress["batch_id"] == batch_id,
            "assembly_accession",
        ]
        .astype(str)
        .tolist()
    )

    observed_batch_accessions = (
        batch_manifest["assembly_accession"]
        .astype(str)
        .tolist()
    )

    if (
        observed_batch_accessions
        != expected_batch_accessions
    ):
        raise ValueError(
            f"{archive_path.name} does not match the "
            "canonical Model C pathogen order."
        )

    duplicated_accessions = (
        represented_accessions
        .intersection(observed_batch_accessions)
    )

    if duplicated_accessions:
        raise ValueError(
            "Assemblies occur in more than one archive: "
            f"{sorted(duplicated_accessions)}"
        )

    represented_accessions.update(
        observed_batch_accessions
    )

    if (
        batch_summary.get("validation_status")
        != "passed"
    ):
        raise ValueError(
            f"{archive_path.name} did not report a passed "
            "validation status."
        )

    if int(
        batch_summary.get("assemblies", -1)
    ) != len(expected_batch_accessions):
        raise ValueError(
            f"{archive_path.name} has an incorrect assembly "
            "count in its summary."
        )

    archive_audit_rows.append(
        {
            "batch_id": batch_id,
            "archive_name": archive_path.name,
            "assemblies": len(batch_manifest),
            "sequence_records": int(
                batch_summary.get(
                    "sequence_records",
                    0,
                )
            ),
            "accepted_sequences": int(
                batch_summary.get(
                    "accepted_sequence_records",
                    0,
                )
            ),
            "review_items": int(
                batch_summary.get(
                    "review_items",
                    0,
                )
            ),
            "archive_size_MB": round(
                archive_path.stat().st_size
                / (1024 ** 2),
                3,
            ),
            "validation_status": "passed",
        }
    )

archive_audit = pd.DataFrame(
    archive_audit_rows
).sort_values("batch_id")

missing_batch_ids = sorted(
    set(range(1, EXPECTED_BATCHES + 1))
    - set(available_batch_ids)
)

audit_summary = pd.DataFrame(
    [
        {
            "metric": "Canonical Model C pathogens",
            "value": len(progress),
        },
        {
            "metric": "Available batch archives",
            "value": len(archive_audit),
        },
        {
            "metric": "Assemblies represented",
            "value": len(represented_accessions),
        },
        {
            "metric": "Duplicate assemblies",
            "value": 0,
        },
        {
            "metric": "Missing batch archives",
            "value": len(missing_batch_ids),
        },
        {
            "metric": "All available archives passed",
            "value": True,
        },
    ]
)

display(audit_summary)
display(archive_audit.tail(10))

ARCHIVE_AUDIT_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_nb23_batch_archive_audit.csv"
)

archive_audit.to_csv(
    ARCHIVE_AUDIT_PATH,
    index=False,
)

print(f"Saved: {ARCHIVE_AUDIT_PATH}")

if missing_batch_ids:
    print(
        "\nNotebook 23 extraction is still incomplete. "
        f"The first missing batch is "
        f"{missing_batch_ids[0]:04d}."
    )
else:
    print(
        "\nAll 906 Notebook 23 batch archives are present "
        "and validated."
    )

print(
    "\nTransition: Cell 24.4 will combine the validated "
    "batch manifests, sequence records and review records."
)

,metric,value
0,Canonical Model C pathogens,9058
1,Available batch archives,906
2,Assemblies represented,9058
3,Duplicate assemblies,0
4,Missing batch archives,0
5,All available archives passed,True


,batch_id,archive_name,assemblies,sequence_records,accepted_sequences,review_items,archive_size_MB,validation_status
896,897,23_batch_0897.zip,10,425,422,8,0.324,passed
897,898,23_batch_0898.zip,10,476,470,8,0.340,passed
898,899,23_batch_0899.zip,10,488,483,12,0.344,passed
899,900,23_batch_0900.zip,10,476,475,4,0.341,passed
900,901,23_batch_0901.zip,10,501,498,14,0.350,passed
901,902,23_batch_0902.zip,10,425,420,11,0.324,passed
902,903,23_batch_0903.zip,10,405,403,4,0.319,passed
903,904,23_batch_0904.zip,10,418,415,7,0.322,passed
904,905,23_batch_0905.zip,10,590,584,10,0.374,passed
905,906,23_batch_0906.zip,8,440,435,10,0.289,passed


Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_nb23_batch_archive_audit.csv

All 906 Notebook 23 batch archives are present and validated.

Transition: Cell 24.4 will combine the validated batch manifests, sequence records and review records.


In [4]:
#@title Cell 24.4 - Consolidate validated Notebook 23 records
# This cell reads the validated batch archives and combines their pathogen
# manifests, extracted sequences and review records. Notebook 23 files are
# read only; the combined files are saved in the Notebook 24 results folder.

def read_archive_csv(
    archive,
    member_name,
):
    data = archive.read(member_name)

    if not data.strip():
        return pd.DataFrame()

    try:
        return pd.read_csv(
            io.BytesIO(data),
            dtype={
                "biosample": str,
                "assembly_accession": str,
                "locus_id": str,
            },
        )
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def convert_to_boolean(value):
    if isinstance(value, bool):
        return value

    text = str(value).strip().lower()

    if text in {"true", "1", "yes"}:
        return True

    if text in {"false", "0", "no"}:
        return False

    raise ValueError(
        f"Unrecognised Boolean value: {value}"
    )


manifest_tables = []
sequence_tables = []
review_tables = []

for audit_row in archive_audit.itertuples(
    index=False
):
    archive_path = (
        NOTEBOOK23_BATCH_DIRECTORY
        / audit_row.archive_name
    )

    with zipfile.ZipFile(
        archive_path,
        "r",
    ) as archive:
        batch_manifest = read_archive_csv(
            archive,
            "batch_manifest.csv",
        )

        batch_sequences = read_archive_csv(
            archive,
            "target_sequences.csv",
        )

        batch_reviews = read_archive_csv(
            archive,
            "review_items.csv",
        )

    batch_manifest.insert(
        0,
        "batch_id",
        int(audit_row.batch_id),
    )

    manifest_tables.append(
        batch_manifest
    )

    if not batch_sequences.empty:
        batch_sequences.insert(
            0,
            "batch_id",
            int(audit_row.batch_id),
        )

        sequence_tables.append(
            batch_sequences
        )

    if not batch_reviews.empty:
        batch_reviews.insert(
            0,
            "batch_id",
            int(audit_row.batch_id),
        )

        review_tables.append(
            batch_reviews
        )

consolidated_manifest = pd.concat(
    manifest_tables,
    ignore_index=True,
    sort=False,
)

consolidated_sequences = (
    pd.concat(
        sequence_tables,
        ignore_index=True,
        sort=False,
    )
    if sequence_tables
    else pd.DataFrame()
)

consolidated_reviews = (
    pd.concat(
        review_tables,
        ignore_index=True,
        sort=False,
    )
    if review_tables
    else pd.DataFrame()
)

# ---------------------------------------------------------------------------
# Validate the consolidated pathogen manifest
# ---------------------------------------------------------------------------

consolidated_manifest[
    "model_c_row_index"
] = pd.to_numeric(
    consolidated_manifest[
        "model_c_row_index"
    ],
    errors="raise",
).astype(int)

consolidated_manifest = (
    consolidated_manifest
    .sort_values("model_c_row_index")
    .reset_index(drop=True)
)

if consolidated_manifest[
    "assembly_accession"
].duplicated().any():
    raise ValueError(
        "An assembly occurs in more than one validated batch."
    )

expected_available_accessions = set(
    progress.loc[
        progress["batch_id"].isin(
            available_batch_ids
        ),
        "assembly_accession",
    ]
)

observed_available_accessions = set(
    consolidated_manifest[
        "assembly_accession"
    ]
)

if (
    observed_available_accessions
    != expected_available_accessions
):
    raise ValueError(
        "The consolidated manifest does not match the "
        "available validated batches."
    )

# ---------------------------------------------------------------------------
# Validate and index the sequence records
# ---------------------------------------------------------------------------

required_sequence_columns = {
    "biosample",
    "assembly_accession",
    "locus_id",
    "locus_name",
    "sequence",
    "accepted_for_kernel",
    "sequence_status",
    "sequence_id",
}

if not consolidated_sequences.empty:
    missing_sequence_columns = (
        required_sequence_columns
        - set(consolidated_sequences.columns)
    )

    if missing_sequence_columns:
        raise ValueError(
            "The consolidated sequence table is missing "
            f"columns: {sorted(missing_sequence_columns)}"
        )

    consolidated_sequences[
        "accepted_for_kernel"
    ] = consolidated_sequences[
        "accepted_for_kernel"
    ].map(convert_to_boolean)

    consolidated_sequences["sequence"] = (
        consolidated_sequences["sequence"]
        .astype(str)
        .str.upper()
    )

    if consolidated_sequences[
        "sequence_id"
    ].duplicated().any():
        raise ValueError(
            "Duplicate sequence identifiers were found."
        )

    unknown_sequence_accessions = (
        set(
            consolidated_sequences[
                "assembly_accession"
            ]
        )
        - observed_available_accessions
    )

    if unknown_sequence_accessions:
        raise ValueError(
            "Sequence records contain assemblies that are "
            "absent from the consolidated manifest."
        )

    locus_panel = pd.read_csv(
        LOCUS_PANEL_PATH,
        dtype={
            "locus_id": str,
            "locus_name": str,
        },
    )

    if len(locus_panel) != EXPECTED_LOCI:
        raise ValueError(
            f"Expected {EXPECTED_LOCI} loci, but the locus "
            f"panel contains {len(locus_panel)}."
        )

    valid_locus_ids = set(
        locus_panel["locus_id"]
    )

    unknown_locus_ids = (
        set(
            consolidated_sequences[
                "locus_id"
            ]
        )
        - valid_locus_ids
    )

    if unknown_locus_ids:
        raise ValueError(
            "Sequence records contain unknown loci: "
            f"{sorted(unknown_locus_ids)}"
        )

    assembly_to_row_index = dict(
        zip(
            consolidated_manifest[
                "assembly_accession"
            ],
            consolidated_manifest[
                "model_c_row_index"
            ],
        )
    )

    consolidated_sequences.insert(
        1,
        "model_c_row_index",
        consolidated_sequences[
            "assembly_accession"
        ].map(assembly_to_row_index),
    )

    consolidated_sequences = (
        consolidated_sequences
        .sort_values(
            [
                "model_c_row_index",
                "locus_id",
                "copy_index",
            ]
        )
        .reset_index(drop=True)
    )

# ---------------------------------------------------------------------------
# Validate and index the review records
# ---------------------------------------------------------------------------

if not consolidated_reviews.empty:
    unknown_review_accessions = (
        set(
            consolidated_reviews[
                "assembly_accession"
            ]
        )
        - observed_available_accessions
    )

    if unknown_review_accessions:
        raise ValueError(
            "Review records contain assemblies that are "
            "absent from the consolidated manifest."
        )

    consolidated_reviews.insert(
        1,
        "model_c_row_index",
        consolidated_reviews[
            "assembly_accession"
        ].map(assembly_to_row_index),
    )

    consolidated_reviews = (
        consolidated_reviews
        .sort_values(
            [
                "model_c_row_index",
                "locus_id",
                "review_type",
            ]
        )
        .reset_index(drop=True)
    )

# ---------------------------------------------------------------------------
# Save the consolidated Notebook 24 inputs
# ---------------------------------------------------------------------------

CONSOLIDATED_MANIFEST_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_consolidated_batch_manifest.csv"
)

CONSOLIDATED_SEQUENCE_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_consolidated_target_sequences.csv.gz"
)

CONSOLIDATED_REVIEW_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_consolidated_review_items.csv"
)

consolidated_manifest.to_csv(
    CONSOLIDATED_MANIFEST_PATH,
    index=False,
)

consolidated_sequences.to_csv(
    CONSOLIDATED_SEQUENCE_PATH,
    index=False,
    compression="gzip",
)

consolidated_reviews.to_csv(
    CONSOLIDATED_REVIEW_PATH,
    index=False,
)

consolidation_complete = (
    len(archive_audit) == EXPECTED_BATCHES
    and len(consolidated_manifest)
    == EXPECTED_MODEL_C_PATHOGENS
)

consolidation_summary = pd.DataFrame(
    [
        {
            "metric": "Validated batch archives",
            "value": len(archive_audit),
        },
        {
            "metric": "Consolidated pathogens",
            "value": len(consolidated_manifest),
        },
        {
            "metric": "Extracted sequence records",
            "value": len(consolidated_sequences),
        },
        {
            "metric": "Accepted sequence records",
            "value": int(
                consolidated_sequences[
                    "accepted_for_kernel"
                ].sum()
            )
            if not consolidated_sequences.empty
            else 0,
        },
        {
            "metric": "Unaccepted sequence records",
            "value": int(
                (
                    ~consolidated_sequences[
                        "accepted_for_kernel"
                    ]
                ).sum()
            )
            if not consolidated_sequences.empty
            else 0,
        },
        {
            "metric": "Review records",
            "value": len(consolidated_reviews),
        },
        {
            "metric": "Final consolidation complete",
            "value": consolidation_complete,
        },
    ]
)

display(consolidation_summary)

print(f"Saved: {CONSOLIDATED_MANIFEST_PATH}")
print(f"Saved: {CONSOLIDATED_SEQUENCE_PATH}")
print(f"Saved: {CONSOLIDATED_REVIEW_PATH}")

if not consolidation_complete:
    print(
        "\nThis is an interim consolidation of the currently "
        "available batches. Rerun Cells 24.3 and 24.4 after "
        "all 906 batch archives have been collected."
    )

print(
    "\nTransition: Cell 24.5 will summarise sequence "
    "availability and unresolved results for every locus."
)

,metric,value
0,Validated batch archives,906
1,Consolidated pathogens,9058
2,Extracted sequence records,367486
3,Accepted sequence records,364969
4,Unaccepted sequence records,2517
5,Review records,6368
6,Final consolidation complete,True


Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_consolidated_batch_manifest.csv
Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_consolidated_target_sequences.csv.gz
Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_consolidated_review_items.csv

Transition: Cell 24.5 will summarise sequence availability and unresolved results for every locus.


In [5]:
# =============================================================================
# Cell 24.5
# =============================================================================

#@title Cell 24.5 - Summarise sequence availability by locus
# This cell counts the extracted and accepted sequences for every targeted
# locus and identifies which loci have a static reference sequence.

locus_panel = pd.read_csv(
    LOCUS_PANEL_PATH,
    dtype={
        "locus_id": str,
        "locus_name": str,
    },
)

if len(locus_panel) != EXPECTED_LOCI:
    raise ValueError(
        f"Expected {EXPECTED_LOCI} loci, but the locus panel "
        f"contains {len(locus_panel)}."
    )

if locus_panel["locus_id"].duplicated().any():
    raise ValueError(
        "Duplicate locus identifiers were found in the locus panel."
    )

static_reference_sequences = {}
static_reference_descriptions = {}

for reference_record in SeqIO.parse(
    str(REFERENCE_FASTA_PATH),
    "fasta",
):
    locus_match = re.search(
        r"LOCUS_\d{4}",
        reference_record.description,
    )

    if locus_match is None:
        raise ValueError(
            "A static reference FASTA record does not contain "
            f"a locus identifier: {reference_record.description}"
        )

    locus_id = locus_match.group(0)
    reference_sequence = str(
        reference_record.seq
    ).upper()

    if locus_id in static_reference_sequences:
        raise ValueError(
            f"More than one static reference was found for {locus_id}."
        )

    if not reference_sequence:
        raise ValueError(
            f"The static reference for {locus_id} is empty."
        )

    if not set(reference_sequence) <= {"A", "C", "G", "T"}:
        raise ValueError(
            f"The static reference for {locus_id} contains "
            "an ambiguous nucleotide."
        )

    static_reference_sequences[locus_id] = (
        reference_sequence
    )
    static_reference_descriptions[locus_id] = (
        reference_record.description
    )

all_sequence_counts = (
    consolidated_sequences.groupby("locus_id")
    .size()
    .rename("extracted_sequence_records")
)

accepted_sequences = consolidated_sequences[
    consolidated_sequences["accepted_for_kernel"]
].copy()

accepted_record_counts = (
    accepted_sequences.groupby("locus_id")
    .size()
    .rename("accepted_sequence_records")
)

accepted_pathogen_counts = (
    accepted_sequences.groupby("locus_id")[
        "assembly_accession"
    ]
    .nunique()
    .rename("pathogens_with_accepted_sequence")
)

accepted_copy_counts = (
    accepted_sequences.groupby(
        [
            "assembly_accession",
            "locus_id",
        ]
    )
    .size()
    .rename("accepted_copies")
    .reset_index()
)

multiple_copy_counts = (
    accepted_copy_counts[
        accepted_copy_counts["accepted_copies"] > 1
    ]
    .groupby("locus_id")
    .size()
    .rename("pathogens_with_multiple_accepted_copies")
)

unaccepted_record_counts = (
    consolidated_sequences[
        ~consolidated_sequences["accepted_for_kernel"]
    ]
    .groupby("locus_id")
    .size()
    .rename("unaccepted_sequence_records")
)

locus_availability = locus_panel.copy()

for summary_series in [
    all_sequence_counts,
    accepted_record_counts,
    accepted_pathogen_counts,
    multiple_copy_counts,
    unaccepted_record_counts,
]:
    locus_availability = locus_availability.merge(
        summary_series,
        left_on="locus_id",
        right_index=True,
        how="left",
    )

count_columns = [
    "extracted_sequence_records",
    "accepted_sequence_records",
    "pathogens_with_accepted_sequence",
    "pathogens_with_multiple_accepted_copies",
    "unaccepted_sequence_records",
]

locus_availability[count_columns] = (
    locus_availability[count_columns]
    .fillna(0)
    .astype(int)
)

locus_availability[
    "pathogens_without_accepted_sequence"
] = (
    len(consolidated_manifest)
    - locus_availability[
        "pathogens_with_accepted_sequence"
    ]
)

locus_availability[
    "static_reference_available"
] = locus_availability["locus_id"].isin(
    static_reference_sequences
)

locus_availability[
    "alignment_reference_rule"
] = np.where(
    locus_availability[
        "static_reference_available"
    ],
    "static reference sequence",
    "observed alignment anchor will be selected",
)

LOCUS_AVAILABILITY_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_locus_sequence_availability.csv"
)

locus_availability.to_csv(
    LOCUS_AVAILABILITY_PATH,
    index=False,
)

availability_summary = pd.DataFrame(
    [
        {
            "metric": "Targeted loci",
            "value": len(locus_availability),
        },
        {
            "metric": "Loci with an accepted sequence",
            "value": int(
                (
                    locus_availability[
                        "accepted_sequence_records"
                    ]
                    > 0
                ).sum()
            ),
        },
        {
            "metric": "Static reference sequences",
            "value": len(static_reference_sequences),
        },
        {
            "metric": "Loci requiring an observed alignment anchor",
            "value": int(
                (
                    ~locus_availability[
                        "static_reference_available"
                    ]
                    & (
                        locus_availability[
                            "accepted_sequence_records"
                        ]
                        > 0
                    )
                ).sum()
            ),
        },
        {
            "metric": "Loci with no accepted sequence",
            "value": int(
                (
                    locus_availability[
                        "accepted_sequence_records"
                    ]
                    == 0
                ).sum()
            ),
        },
    ]
)

display(availability_summary)
display(
    locus_availability[
        locus_availability[
            "accepted_sequence_records"
        ]
        == 0
    ][
        [
            "locus_id",
            "locus_name",
            "locus_group",
        ]
    ]
)

print(f"Saved: {LOCUS_AVAILABILITY_PATH}")

print(
    "\nTransition: Cell 24.6 will select one accepted "
    "sequence per pathogen and locus and define the "
    "alignment anchor for every represented locus."
)


,metric,value
0,Targeted loci,297
1,Loci with an accepted sequence,265
2,Static reference sequences,31
3,Loci requiring an observed alignment anchor,234
4,Loci with no accepted sequence,32


,locus_id,locus_name,locus_group
1,LOCUS_0002,aac(3),AMR gene
5,LOCUS_0006,aac(3)-IId=MISTRANSLATION,AMR gene
9,LOCUS_0010,aac(3)-Ib,AMR gene
16,LOCUS_0017,aac(6')-Ib11,AMR gene
35,LOCUS_0036,acrF=MISTRANSLATION,AMR gene
38,LOCUS_0039,ant(3''),AMR gene
40,LOCUS_0041,aph(3''),AMR gene
52,LOCUS_0053,bla,AMR gene
84,LOCUS_0085,blaEC=MISTRANSLATION,AMR gene
91,LOCUS_0092,blaLAP-1,AMR gene


Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_locus_sequence_availability.csv

Transition: Cell 24.6 will select one accepted sequence per pathogen and locus and define the alignment anchor for every represented locus.


In [6]:
# =============================================================================
# Cell 24.6
# =============================================================================

#@title Cell 24.6 - Select pathogen sequences and alignment anchors
# This cell selects one accepted sequence for each pathogen-locus pair.
# A static reference is used when available; otherwise, the most frequently
# observed accepted sequence is used only as a reproducible alignment anchor.

selected_source = accepted_sequences.copy()

for numeric_column in [
    "query_coverage",
    "percent_identity",
    "copy_index",
]:
    if numeric_column not in selected_source.columns:
        selected_source[numeric_column] = np.nan

    selected_source[numeric_column] = pd.to_numeric(
        selected_source[numeric_column],
        errors="coerce",
    )

selected_source[
    "sequence_length_for_ranking"
] = selected_source["sequence"].str.len()

selected_source[
    "coverage_for_ranking"
] = selected_source["query_coverage"].fillna(
    -1.0
)

selected_source[
    "identity_for_ranking"
] = selected_source["percent_identity"].fillna(
    -1.0
)

selected_source[
    "copy_for_ranking"
] = selected_source["copy_index"].fillna(
    np.inf
)

selected_source = selected_source.sort_values(
    [
        "assembly_accession",
        "locus_id",
        "coverage_for_ranking",
        "identity_for_ranking",
        "sequence_length_for_ranking",
        "copy_for_ranking",
        "sequence_id",
    ],
    ascending=[
        True,
        True,
        False,
        False,
        False,
        True,
        True,
    ],
)

selected_sequences = (
    selected_source.drop_duplicates(
        [
            "assembly_accession",
            "locus_id",
        ],
        keep="first",
    )
    .copy()
    .reset_index(drop=True)
)

selected_sequences[
    "accepted_copy_count"
] = selected_sequences.set_index(
    [
        "assembly_accession",
        "locus_id",
    ]
).index.map(
    accepted_copy_counts.set_index(
        [
            "assembly_accession",
            "locus_id",
        ]
    )["accepted_copies"]
)

selected_sequences[
    "selection_rule"
] = (
    "highest coverage, then highest identity, "
    "then longest sequence"
)

selected_sequences[
    "alignment_record_id"
] = selected_sequences[
    "model_c_row_index"
].map(
    lambda value: f"P{int(value):05d}"
)

if selected_sequences.duplicated(
    [
        "assembly_accession",
        "locus_id",
    ]
).any():
    raise ValueError(
        "More than one sequence remains for a pathogen-locus pair."
    )

if selected_sequences.duplicated(
    [
        "alignment_record_id",
        "locus_id",
    ]
).any():
    raise ValueError(
        "Duplicate alignment record identifiers were generated."
    )

invalid_selected_sequences = selected_sequences[
    ~selected_sequences["sequence"].map(
        lambda sequence: (
            bool(sequence)
            and set(sequence)
            <= {"A", "C", "G", "T"}
        )
    )
]

if not invalid_selected_sequences.empty:
    raise ValueError(
        "An accepted sequence contains an ambiguous nucleotide."
    )

anchor_rows = []

for locus in locus_panel.itertuples(
    index=False
):
    locus_id = locus.locus_id
    locus_sequences = selected_sequences[
        selected_sequences["locus_id"] == locus_id
    ]

    if locus_id in static_reference_sequences:
        anchor_sequence = (
            static_reference_sequences[locus_id]
        )
        anchor_type = "static reference sequence"
        anchor_source = (
            static_reference_descriptions[locus_id]
        )
        anchor_observation_count = np.nan

    elif not locus_sequences.empty:
        observed_variants = (
            locus_sequences.groupby("sequence")
            .agg(
                observation_count=(
                    "assembly_accession",
                    "size",
                ),
                sequence_length=(
                    "sequence",
                    lambda values: len(
                        values.iloc[0]
                    ),
                ),
                first_sequence_id=(
                    "sequence_id",
                    "min",
                ),
            )
            .reset_index()
            .sort_values(
                [
                    "observation_count",
                    "sequence_length",
                    "first_sequence_id",
                ],
                ascending=[
                    False,
                    False,
                    True,
                ],
            )
        )

        selected_anchor = observed_variants.iloc[0]
        anchor_sequence = selected_anchor["sequence"]
        anchor_type = "observed alignment anchor"
        anchor_source = selected_anchor[
            "first_sequence_id"
        ]
        anchor_observation_count = int(
            selected_anchor["observation_count"]
        )

    else:
        anchor_sequence = ""
        anchor_type = "no alignment anchor"
        anchor_source = ""
        anchor_observation_count = 0

    anchor_rows.append(
        {
            "locus_id": locus_id,
            "locus_name": locus.locus_name,
            "anchor_type": anchor_type,
            "anchor_source": anchor_source,
            "anchor_observation_count":
                anchor_observation_count,
            "anchor_sequence_length":
                len(anchor_sequence),
            "anchor_sequence": anchor_sequence,
            "selected_pathogen_sequences":
                len(locus_sequences),
        }
    )

alignment_anchors = pd.DataFrame(
    anchor_rows
)

SELECTED_SEQUENCE_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_selected_pathogen_sequences.csv.gz"
)

ALIGNMENT_ANCHOR_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_alignment_anchors.csv"
)

ALIGNMENT_ANCHOR_FASTA_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_alignment_anchors.fasta"
)

selected_sequences.to_csv(
    SELECTED_SEQUENCE_PATH,
    index=False,
    compression="gzip",
)

alignment_anchors.to_csv(
    ALIGNMENT_ANCHOR_PATH,
    index=False,
)

with open(
    ALIGNMENT_ANCHOR_FASTA_PATH,
    "w",
    encoding="utf-8",
) as fasta_file:
    for anchor in alignment_anchors.itertuples(
        index=False
    ):
        if not anchor.anchor_sequence:
            continue

        fasta_file.write(
            f">{anchor.locus_id}|{anchor.locus_name}|"
            f"{anchor.anchor_type.replace(' ', '_')}\n"
        )

        for start in range(
            0,
            len(anchor.anchor_sequence),
            80,
        ):
            fasta_file.write(
                anchor.anchor_sequence[
                    start:start + 80
                ]
                + "\n"
            )

selection_summary = pd.DataFrame(
    [
        {
            "metric": "Accepted pathogen-locus records",
            "value": len(selected_sequences),
        },
        {
            "metric": "Pathogen-locus pairs with multiple accepted copies",
            "value": int(
                (
                    selected_sequences[
                        "accepted_copy_count"
                    ]
                    > 1
                ).sum()
            ),
        },
        {
            "metric": "Static reference anchors",
            "value": int(
                (
                    alignment_anchors[
                        "anchor_type"
                    ]
                    == "static reference sequence"
                ).sum()
            ),
        },
        {
            "metric": "Observed alignment anchors",
            "value": int(
                (
                    alignment_anchors[
                        "anchor_type"
                    ]
                    == "observed alignment anchor"
                ).sum()
            ),
        },
        {
            "metric": "Loci with no alignment anchor",
            "value": int(
                (
                    alignment_anchors[
                        "anchor_type"
                    ]
                    == "no alignment anchor"
                ).sum()
            ),
        },
    ]
)

display(selection_summary)

print(
    "\nAn observed alignment anchor is a computational "
    "reference for alignment. It is not described as a "
    "biological wild-type sequence."
)

print(f"Saved: {SELECTED_SEQUENCE_PATH}")
print(f"Saved: {ALIGNMENT_ANCHOR_PATH}")
print(f"Saved: {ALIGNMENT_ANCHOR_FASTA_PATH}")

print(
    "\nTransition: Cell 24.7 will install MAFFT and "
    "validate nucleotide-sequence alignment on a small test."
)


,metric,value
0,Accepted pathogen-locus records,333286
1,Pathogen-locus pairs with multiple accepted co...,28963
2,Static reference anchors,31
3,Observed alignment anchors,234
4,Loci with no alignment anchor,32



An observed alignment anchor is a computational reference for alignment. It is not described as a biological wild-type sequence.
Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_selected_pathogen_sequences.csv.gz
Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_alignment_anchors.csv
Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_alignment_anchors.fasta

Transition: Cell 24.7 will install MAFFT and validate nucleotide-sequence alignment on a small test.


In [7]:
# =============================================================================
# Cell 24.7
# =============================================================================

#@title Cell 24.7 - Install and test MAFFT
# This cell installs the MAFFT nucleotide-alignment program when necessary
# and confirms that it produces a valid alignment on a small sequence subset.

import gzip
import os
import tempfile
import time

WORK_DIRECTORY = Path(
    "/content/notebook24_work"
)

WORK_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

MAFFT_EXECUTABLE = shutil.which("mafft")

if MAFFT_EXECUTABLE is None:
    print("Installing MAFFT...")

    subprocess.run(
        [
            "apt-get",
            "update",
            "-qq",
        ],
        check=True,
    )

    subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            "-qq",
            "mafft",
        ],
        check=True,
    )

    MAFFT_EXECUTABLE = shutil.which("mafft")

if MAFFT_EXECUTABLE is None:
    raise FileNotFoundError(
        "MAFFT was not installed successfully."
    )

version_result = subprocess.run(
    [
        MAFFT_EXECUTABLE,
        "--version",
    ],
    capture_output=True,
    text=True,
    check=False,
)

mafft_version = (
    version_result.stderr.strip()
    or version_result.stdout.strip()
)

test_candidates = alignment_anchors[
    alignment_anchors[
        "selected_pathogen_sequences"
    ].between(2, 20)
]

if test_candidates.empty:
    test_candidates = alignment_anchors[
        alignment_anchors[
            "selected_pathogen_sequences"
        ]
        >= 2
    ]

if test_candidates.empty:
    raise ValueError(
        "No locus has enough accepted sequences for "
        "the MAFFT test."
    )

test_anchor = test_candidates.sort_values(
    "selected_pathogen_sequences"
).iloc[0]

test_locus_id = test_anchor["locus_id"]

test_sequences = (
    selected_sequences[
        selected_sequences["locus_id"]
        == test_locus_id
    ]
    .sort_values("model_c_row_index")
    .head(5)
)

test_input_path = (
    WORK_DIRECTORY
    / "24_mafft_test_input.fasta"
)

test_output_path = (
    WORK_DIRECTORY
    / "24_mafft_test_output.fasta"
)

with open(
    test_input_path,
    "w",
    encoding="utf-8",
) as fasta_file:
    fasta_file.write(
        f">ANCHOR\n{test_anchor['anchor_sequence']}\n"
    )

    for sequence_row in test_sequences.itertuples(
        index=False
    ):
        fasta_file.write(
            f">{sequence_row.alignment_record_id}\n"
            f"{sequence_row.sequence}\n"
        )

with open(
    test_output_path,
    "w",
    encoding="utf-8",
) as output_file:
    test_result = subprocess.run(
        [
            MAFFT_EXECUTABLE,
            "--nuc",
            "--auto",
            "--inputorder",
            "--thread",
            "2",
            str(test_input_path),
        ],
        stdout=output_file,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )

if test_result.returncode != 0:
    raise RuntimeError(
        "The MAFFT test failed:\n"
        f"{test_result.stderr[-3000:]}"
    )

test_alignment = list(
    SeqIO.parse(
        str(test_output_path),
        "fasta",
    )
)

expected_test_ids = {
    "ANCHOR",
    *test_sequences[
        "alignment_record_id"
    ].tolist(),
}

observed_test_ids = {
    record.id
    for record in test_alignment
}

if observed_test_ids != expected_test_ids:
    raise ValueError(
        "The MAFFT test alignment contains unexpected "
        "sequence identifiers."
    )

test_alignment_lengths = {
    len(record.seq)
    for record in test_alignment
}

if len(test_alignment_lengths) != 1:
    raise ValueError(
        "The MAFFT test did not produce equal sequence lengths."
    )

for record in test_alignment:
    aligned_sequence = str(record.seq).upper()

    if not set(aligned_sequence) <= {
        "A",
        "C",
        "G",
        "T",
        "-",
    }:
        raise ValueError(
            "The MAFFT test produced an unexpected "
            "alignment character."
        )

display(
    pd.DataFrame(
        [
            {
                "metric": "MAFFT version",
                "value": mafft_version,
            },
            {
                "metric": "Test locus",
                "value": test_locus_id,
            },
            {
                "metric": "Sequences aligned",
                "value": len(test_alignment),
            },
            {
                "metric": "Aligned positions",
                "value": next(
                    iter(test_alignment_lengths)
                ),
            },
            {
                "metric": "MAFFT test status",
                "value": "passed",
            },
        ]
    )
)

test_input_path.unlink(
    missing_ok=True
)

test_output_path.unlink(
    missing_ok=True
)

print(
    "MAFFT installation and nucleotide-alignment "
    "validation passed."
)

print(
    "\nTransition: Cell 24.8 will align every represented "
    "locus and save one restartable checkpoint per locus."
)

Installing MAFFT...


,metric,value
0,MAFFT version,v7.490 (2021/Oct/30)
1,Test locus,LOCUS_0001
2,Sequences aligned,3
3,Aligned positions,780
4,MAFFT test status,passed


MAFFT installation and nucleotide-alignment validation passed.

Transition: Cell 24.8 will align every represented locus and save one restartable checkpoint per locus.


In [8]:
# =============================================================================
# Cell 24.8
# =============================================================================

#@title Cell 24.8 - Align all represented loci with restartable checkpoints
# This cell aligns the selected pathogen sequences separately for each locus.
# Every completed locus is validated and saved before the next locus begins,
# so rerunning the cell skips valid completed alignment checkpoints.

if not consolidation_complete:
    raise RuntimeError(
        "Final alignment must wait until all 906 Notebook 23 "
        "batch archives have been consolidated."
    )

ALIGNMENT_CHECKPOINT_DIRECTORY = (
    NOTEBOOK24_CHECKPOINT_DIRECTORY
)

ALIGNMENT_CHECKPOINT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

LOCAL_ALIGNMENT_DIRECTORY = (
    WORK_DIRECTORY / "alignments"
)

LOCAL_ALIGNMENT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


def sequence_input_sha256(
    records,
):
    digest = hashlib.sha256()

    for record_id, sequence in records:
        digest.update(
            f">{record_id}\n{sequence}\n".encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def file_sha256(
    file_path,
):
    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as file_handle:
        for chunk in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def load_gzip_alignment(
    alignment_path,
):
    with gzip.open(
        alignment_path,
        "rt",
        encoding="utf-8",
    ) as file_handle:
        return list(
            SeqIO.parse(
                file_handle,
                "fasta",
            )
        )


def validate_alignment_records(
    alignment_records,
    expected_ids,
):
    observed_ids = [
        record.id
        for record in alignment_records
    ]

    if len(observed_ids) != len(
        set(observed_ids)
    ):
        raise ValueError(
            "An alignment contains duplicate identifiers."
        )

    if set(observed_ids) != set(expected_ids):
        raise ValueError(
            "An alignment does not contain the expected "
            "sequence identifiers."
        )

    aligned_lengths = {
        len(record.seq)
        for record in alignment_records
    }

    if len(aligned_lengths) != 1:
        raise ValueError(
            "Aligned sequences do not have equal lengths."
        )

    for record in alignment_records:
        aligned_sequence = str(record.seq).upper()

        if not set(aligned_sequence) <= {
            "A",
            "C",
            "G",
            "T",
            "-",
        }:
            raise ValueError(
                "An alignment contains an unexpected symbol."
            )

        if not set(aligned_sequence) & {
            "A",
            "C",
            "G",
            "T",
        }:
            raise ValueError(
                "An aligned sequence contains only gaps."
            )

    return next(iter(aligned_lengths))


alignment_checkpoint_rows = []

for locus_number, anchor in enumerate(
    alignment_anchors.itertuples(
        index=False
    ),
    start=1,
):
    locus_id = anchor.locus_id

    locus_sequences = (
        selected_sequences[
            selected_sequences["locus_id"]
            == locus_id
        ]
        .sort_values("model_c_row_index")
    )

    if (
        not anchor.anchor_sequence
        or locus_sequences.empty
    ):
        alignment_checkpoint_rows.append(
            {
                "locus_id": locus_id,
                "locus_name": anchor.locus_name,
                "pathogen_sequences": 0,
                "aligned_positions": 0,
                "checkpoint_status":
                    "no accepted sequence",
            }
        )
        continue

    input_records = [
        (
            "ANCHOR",
            anchor.anchor_sequence,
        )
    ]

    input_records.extend(
        zip(
            locus_sequences[
                "alignment_record_id"
            ],
            locus_sequences["sequence"],
        )
    )

    input_records = list(input_records)

    input_sha256 = sequence_input_sha256(
        input_records
    )

    checkpoint_path = (
        ALIGNMENT_CHECKPOINT_DIRECTORY
        / f"24_{locus_id}_alignment.fasta.gz"
    )

    metadata_path = (
        ALIGNMENT_CHECKPOINT_DIRECTORY
        / f"24_{locus_id}_alignment.json"
    )

    expected_ids = [
        record_id
        for record_id, _ in input_records
    ]

    if (
        checkpoint_path.exists()
        and metadata_path.exists()
    ):
        with open(
            metadata_path,
            "r",
            encoding="utf-8",
        ) as metadata_file:
            saved_metadata = json.load(
                metadata_file
            )

        if (
            saved_metadata.get("input_sha256")
            != input_sha256
        ):
            raise ValueError(
                f"The saved {locus_id} alignment was made "
                "from different input sequences."
            )

        if (
            saved_metadata.get("alignment_sha256")
            != file_sha256(checkpoint_path)
        ):
            raise ValueError(
                f"The saved {locus_id} alignment checksum "
                "does not match its metadata."
            )

        saved_alignment = load_gzip_alignment(
            checkpoint_path
        )

        aligned_positions = (
            validate_alignment_records(
                saved_alignment,
                expected_ids,
            )
        )

        alignment_checkpoint_rows.append(
            {
                "locus_id": locus_id,
                "locus_name": anchor.locus_name,
                "pathogen_sequences":
                    len(locus_sequences),
                "aligned_positions":
                    aligned_positions,
                "checkpoint_status":
                    "existing checkpoint validated",
            }
        )

        print(
            f"[{locus_number}/{EXPECTED_LOCI}] "
            f"{locus_id}: existing checkpoint validated"
        )
        continue

    if (
        checkpoint_path.exists()
        != metadata_path.exists()
    ):
        raise FileNotFoundError(
            f"The {locus_id} alignment checkpoint is incomplete."
        )

    local_input_path = (
        LOCAL_ALIGNMENT_DIRECTORY
        / f"{locus_id}_input.fasta"
    )

    local_output_path = (
        LOCAL_ALIGNMENT_DIRECTORY
        / f"{locus_id}_alignment.fasta"
    )

    with open(
        local_input_path,
        "w",
        encoding="utf-8",
    ) as fasta_file:
        for record_id, sequence in input_records:
            fasta_file.write(
                f">{record_id}\n"
            )

            for start in range(
                0,
                len(sequence),
                80,
            ):
                fasta_file.write(
                    sequence[start:start + 80]
                    + "\n"
                )

    started = time.perf_counter()

    with open(
        local_output_path,
        "w",
        encoding="utf-8",
    ) as output_file:
        alignment_result = subprocess.run(
            [
                MAFFT_EXECUTABLE,
                "--nuc",
                "--auto",
                "--inputorder",
                "--thread",
                "2",
                str(local_input_path),
            ],
            stdout=output_file,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
        )

    if alignment_result.returncode != 0:
        raise RuntimeError(
            f"MAFFT failed for {locus_id}:\n"
            f"{alignment_result.stderr[-3000:]}"
        )

    aligned_records = list(
        SeqIO.parse(
            str(local_output_path),
            "fasta",
        )
    )

    aligned_positions = validate_alignment_records(
        aligned_records,
        expected_ids,
    )

    local_gzip_path = (
        LOCAL_ALIGNMENT_DIRECTORY
        / f"{locus_id}_alignment.fasta.gz"
    )

    with open(
        local_output_path,
        "rb",
    ) as input_file:
        with gzip.open(
            local_gzip_path,
            "wb",
            compresslevel=6,
        ) as output_file:
            shutil.copyfileobj(
                input_file,
                output_file,
            )

    partial_checkpoint_path = (
        checkpoint_path.with_suffix(
            checkpoint_path.suffix + ".partial"
        )
    )

    partial_checkpoint_path.unlink(
        missing_ok=True
    )

    shutil.copy2(
        local_gzip_path,
        partial_checkpoint_path,
    )

    partial_checkpoint_path.replace(
        checkpoint_path
    )

    alignment_sha256 = file_sha256(
        checkpoint_path
    )

    metadata = {
        "locus_id": locus_id,
        "locus_name": anchor.locus_name,
        "anchor_type": anchor.anchor_type,
        "input_sha256": input_sha256,
        "alignment_sha256": alignment_sha256,
        "pathogen_sequences":
            len(locus_sequences),
        "alignment_records":
            len(aligned_records),
        "aligned_positions":
            aligned_positions,
        "mafft_version": mafft_version,
        "elapsed_minutes": round(
            (
                time.perf_counter()
                - started
            )
            / 60.0,
            3,
        ),
    }

    temporary_metadata_path = (
        metadata_path.with_suffix(
            ".json.partial"
        )
    )

    with open(
        temporary_metadata_path,
        "w",
        encoding="utf-8",
    ) as metadata_file:
        json.dump(
            metadata,
            metadata_file,
            indent=2,
        )

    temporary_metadata_path.replace(
        metadata_path
    )

    alignment_checkpoint_rows.append(
        {
            "locus_id": locus_id,
            "locus_name": anchor.locus_name,
            "pathogen_sequences":
                len(locus_sequences),
            "aligned_positions":
                aligned_positions,
            "checkpoint_status":
                "saved and validated",
        }
    )

    local_input_path.unlink(
        missing_ok=True
    )
    local_output_path.unlink(
        missing_ok=True
    )
    local_gzip_path.unlink(
        missing_ok=True
    )

    print(
        f"[{locus_number}/{EXPECTED_LOCI}] "
        f"{locus_id}: {len(locus_sequences):,} sequences, "
        f"{aligned_positions:,} aligned positions"
    )

alignment_checkpoint_status = pd.DataFrame(
    alignment_checkpoint_rows
)

ALIGNMENT_CHECKPOINT_STATUS_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_alignment_checkpoint_status.csv"
)

alignment_checkpoint_status.to_csv(
    ALIGNMENT_CHECKPOINT_STATUS_PATH,
    index=False,
)

display(
    alignment_checkpoint_status[
        "checkpoint_status"
    ].value_counts().rename_axis(
        "checkpoint_status"
    ).reset_index(
        name="loci"
    )
)

print(
    f"Saved: {ALIGNMENT_CHECKPOINT_STATUS_PATH}"
)

print(
    "\nTransition: Cell 24.9 will validate every completed "
    "alignment and calculate the total aligned dimension."
)



[1/297] LOCUS_0001: 2 sequences, 780 aligned positions
[3/297] LOCUS_0003: 1 sequences, 792 aligned positions
[4/297] LOCUS_0004: 2 sequences, 858 aligned positions
[5/297] LOCUS_0005: 427 sequences, 858 aligned positions
[7/297] LOCUS_0007: 266 sequences, 858 aligned positions
[8/297] LOCUS_0008: 2 sequences, 807 aligned positions
[9/297] LOCUS_0009: 123 sequences, 774 aligned positions
[11/297] LOCUS_0011: 598 sequences, 897 aligned positions
[12/297] LOCUS_0012: 2 sequences, 579 aligned positions
[13/297] LOCUS_0013: 3 sequences, 552 aligned positions
[14/297] LOCUS_0014: 3 sequences, 552 aligned positions
[15/297] LOCUS_0015: 18 sequences, 552 aligned positions
[16/297] LOCUS_0016: 342 sequences, 552 aligned positions
[18/297] LOCUS_0018: 12 sequences, 552 aligned positions
[19/297] LOCUS_0019: 6 sequences, 552 aligned positions
[20/297] LOCUS_0020: 1 sequences, 456 aligned positions
[21/297] LOCUS_0021: 1,355 sequences, 819 aligned positions
[22/297] LOCUS_0022: 26 sequences, 789 

,checkpoint_status,loci
0,saved and validated,265
1,no accepted sequence,32


Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_alignment_checkpoint_status.csv

Transition: Cell 24.9 will validate every completed alignment and calculate the total aligned dimension.


In [9]:
# =============================================================================
# Cell 24.9
# =============================================================================

#@title Cell 24.9 - Validate locus alignments and aligned dimensions
# This cell confirms that each aligned pathogen sequence reproduces its
# original nucleotide sequence after alignment gaps are removed. It then
# records each locus length and the total five-state sequence dimension.

alignment_validation_rows = []
selected_sequence_lookup = (
    selected_sequences.set_index(
        [
            "locus_id",
            "alignment_record_id",
        ]
    )["sequence"].to_dict()
)

for anchor in alignment_anchors.itertuples(
    index=False
):
    locus_id = anchor.locus_id

    locus_sequences = selected_sequences[
        selected_sequences["locus_id"]
        == locus_id
    ]

    if (
        not anchor.anchor_sequence
        or locus_sequences.empty
    ):
        alignment_validation_rows.append(
            {
                "locus_id": locus_id,
                "locus_name": anchor.locus_name,
                "anchor_type": anchor.anchor_type,
                "pathogen_sequences": 0,
                "unique_aligned_sequences": 0,
                "anchor_sequence_length":
                    len(anchor.anchor_sequence),
                "aligned_positions": 0,
                "five_state_dimensions": 0,
                "alignment_sha256": "",
                "validation_status":
                    "no accepted sequence",
            }
        )
        continue

    checkpoint_path = (
        ALIGNMENT_CHECKPOINT_DIRECTORY
        / f"24_{locus_id}_alignment.fasta.gz"
    )

    metadata_path = (
        ALIGNMENT_CHECKPOINT_DIRECTORY
        / f"24_{locus_id}_alignment.json"
    )

    if (
        not checkpoint_path.exists()
        or not metadata_path.exists()
    ):
        raise FileNotFoundError(
            f"The final alignment checkpoint for "
            f"{locus_id} is missing."
        )

    with open(
        metadata_path,
        "r",
        encoding="utf-8",
    ) as metadata_file:
        metadata = json.load(
            metadata_file
        )

    if (
        metadata.get("alignment_sha256")
        != file_sha256(checkpoint_path)
    ):
        raise ValueError(
            f"The saved {locus_id} alignment checksum "
            "does not match its metadata."
        )

    alignment_records = load_gzip_alignment(
        checkpoint_path
    )

    expected_ids = {
        "ANCHOR",
        *locus_sequences[
            "alignment_record_id"
        ].tolist(),
    }

    aligned_positions = validate_alignment_records(
        alignment_records,
        expected_ids,
    )

    observed_anchor = next(
        record
        for record in alignment_records
        if record.id == "ANCHOR"
    )

    if (
        str(observed_anchor.seq)
        .upper()
        .replace("-", "")
        != anchor.anchor_sequence
    ):
        raise ValueError(
            f"The aligned anchor for {locus_id} does not "
            "reproduce its input sequence."
        )

    aligned_pathogen_sequences = []

    for record in alignment_records:
        if record.id == "ANCHOR":
            continue

        original_sequence = selected_sequence_lookup[
            (
                locus_id,
                record.id,
            )
        ]

        aligned_sequence = str(
            record.seq
        ).upper()

        if (
            aligned_sequence.replace("-", "")
            != original_sequence
        ):
            raise ValueError(
                f"The aligned sequence {record.id} for "
                f"{locus_id} does not reproduce its input."
            )

        aligned_pathogen_sequences.append(
            aligned_sequence
        )

    alignment_validation_rows.append(
        {
            "locus_id": locus_id,
            "locus_name": anchor.locus_name,
            "anchor_type": anchor.anchor_type,
            "pathogen_sequences":
                len(aligned_pathogen_sequences),
            "unique_aligned_sequences":
                len(
                    set(
                        aligned_pathogen_sequences
                    )
                ),
            "anchor_sequence_length":
                len(anchor.anchor_sequence),
            "aligned_positions":
                aligned_positions,
            "five_state_dimensions":
                5 * aligned_positions,
            "alignment_sha256":
                metadata["alignment_sha256"],
            "validation_status": "passed",
        }
    )

alignment_validation = pd.DataFrame(
    alignment_validation_rows
)

TOTAL_ALIGNED_POSITIONS = int(
    alignment_validation[
        "aligned_positions"
    ].sum()
)

TOTAL_FIVE_STATE_DIMENSIONS = int(
    alignment_validation[
        "five_state_dimensions"
    ].sum()
)

if (
    TOTAL_FIVE_STATE_DIMENSIONS
    != 5 * TOTAL_ALIGNED_POSITIONS
):
    raise AssertionError(
        "The five-state dimension is inconsistent."
    )

ALIGNMENT_VALIDATION_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_alignment_validation.csv"
)

alignment_validation.to_csv(
    ALIGNMENT_VALIDATION_PATH,
    index=False,
)

display(
    pd.DataFrame(
        [
            {
                "metric": "Targeted loci",
                "value": EXPECTED_LOCI,
            },
            {
                "metric": "Validated locus alignments",
                "value": int(
                    (
                        alignment_validation[
                            "validation_status"
                        ]
                        == "passed"
                    ).sum()
                ),
            },
            {
                "metric": "Loci with no accepted sequence",
                "value": int(
                    (
                        alignment_validation[
                            "validation_status"
                        ]
                        == "no accepted sequence"
                    ).sum()
                ),
            },
            {
                "metric": "Total aligned nucleotide positions",
                "value": TOTAL_ALIGNED_POSITIONS,
            },
            {
                "metric": "Total five-state dimensions",
                "value": TOTAL_FIVE_STATE_DIMENSIONS,
            },
        ]
    )
)

print(f"Saved: {ALIGNMENT_VALIDATION_PATH}")

print(
    "\nTransition: Cell 24.10 will calculate the "
    "9,058 by 9,058 sequence kernel without constructing "
    "the full five-state feature matrix in memory."
)



,metric,value
0,Targeted loci,297
1,Validated locus alignments,265
2,Loci with no accepted sequence,32
3,Total aligned nucleotide positions,247100
4,Total five-state dimensions,1235500


Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_alignment_validation.csv

Transition: Cell 24.10 will calculate the 9,058 by 9,058 sequence kernel without constructing the full five-state feature matrix in memory.


In [10]:
# =============================================================================
# Cell 24.10
# =============================================================================

#@title Cell 24.10 - Calculate the targeted sequence kernel
# This cell counts matching aligned states for every pathogen pair, including
# alignment-gap matches but excluding missing or unreliable locus results.
# It uses local memory-mapped arrays and locus-specific sequence compression.

from scipy.sparse import csr_matrix

KERNEL_BLOCK_SIZE = 128
ALLELE_BLOCK_SIZE = 128

RAW_KERNEL_WORK_PATH = (
    WORK_DIRECTORY
    / "24_raw_sequence_match_counts.dat"
)

LOCAL_KERNEL_PATH = (
    WORK_DIRECTORY
    / "24_sequence_kernel.npy"
)

SEQUENCE_KERNEL_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_sequence_kernel.npy"
)

VALID_POSITION_COUNT_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_valid_aligned_position_counts.npy"
)

KERNEL_METADATA_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_sequence_kernel_metadata.json"
)

alignment_signature = hashlib.sha256(
    "\n".join(
        alignment_validation[
            "alignment_sha256"
        ].fillna("").astype(str)
    ).encode("utf-8")
).hexdigest()

reuse_existing_kernel = False

if (
    SEQUENCE_KERNEL_PATH.exists()
    and VALID_POSITION_COUNT_PATH.exists()
    and KERNEL_METADATA_PATH.exists()
):
    with open(
        KERNEL_METADATA_PATH,
        "r",
        encoding="utf-8",
    ) as metadata_file:
        existing_kernel_metadata = json.load(
            metadata_file
        )

    reuse_existing_kernel = (
        existing_kernel_metadata.get(
            "alignment_signature"
        )
        == alignment_signature
        and existing_kernel_metadata.get(
            "kernel_shape"
        )
        == [
            EXPECTED_MODEL_C_PATHOGENS,
            EXPECTED_MODEL_C_PATHOGENS,
        ]
    )

if reuse_existing_kernel:
    valid_aligned_position_counts = np.load(
        VALID_POSITION_COUNT_PATH
    )

    print(
        "The existing sequence kernel matches the current "
        "validated alignments and will be reused."
    )

else:
    RAW_KERNEL_WORK_PATH.unlink(
        missing_ok=True
    )
    LOCAL_KERNEL_PATH.unlink(
        missing_ok=True
    )

    raw_match_kernel = np.memmap(
        RAW_KERNEL_WORK_PATH,
        dtype=np.uint32,
        mode="w+",
        shape=(
            EXPECTED_MODEL_C_PATHOGENS,
            EXPECTED_MODEL_C_PATHOGENS,
        ),
    )

    raw_match_kernel[:] = 0

    valid_aligned_position_counts = np.zeros(
        EXPECTED_MODEL_C_PATHOGENS,
        dtype=np.uint32,
    )

    state_lookup = np.full(
        256,
        255,
        dtype=np.uint8,
    )

    for state_number, state in enumerate(
        NUCLEOTIDE_STATES
    ):
        state_lookup[ord(state)] = state_number

    aligned_loci = alignment_validation[
        alignment_validation[
            "validation_status"
        ]
        == "passed"
    ]

    for locus_number, locus in enumerate(
        aligned_loci.itertuples(
            index=False
        ),
        start=1,
    ):
        checkpoint_path = (
            ALIGNMENT_CHECKPOINT_DIRECTORY
            / f"24_{locus.locus_id}_alignment.fasta.gz"
        )

        records = load_gzip_alignment(
            checkpoint_path
        )

        pathogen_records = [
            record
            for record in records
            if record.id != "ANCHOR"
        ]

        pathogen_row_indices = np.array(
            [
                int(record.id[1:])
                for record in pathogen_records
            ],
            dtype=np.int32,
        )

        aligned_positions = int(
            locus.aligned_positions
        )

        sequence_bytes = "".join(
            str(record.seq).upper()
            for record in pathogen_records
        ).encode("ascii")

        encoded_states = state_lookup[
            np.frombuffer(
                sequence_bytes,
                dtype=np.uint8,
            )
        ].reshape(
            len(pathogen_records),
            aligned_positions,
        )

        if (
            encoded_states == 255
        ).any():
            raise ValueError(
                f"Unexpected state in {locus.locus_id}."
            )

        unique_states, inverse_alleles = np.unique(
            encoded_states,
            axis=0,
            return_inverse=True,
        )

        unique_alleles = len(
            unique_states
        )

        if aligned_positions > np.iinfo(
            np.uint16
        ).max:
            allele_count_dtype = np.uint32
        else:
            allele_count_dtype = np.uint16

        feature_rows = np.repeat(
            np.arange(
                unique_alleles,
                dtype=np.int32,
            ),
            aligned_positions,
        )

        feature_columns = (
            np.tile(
                np.arange(
                    aligned_positions,
                    dtype=np.int32,
                ),
                unique_alleles,
            )
            * 5
            + unique_states.reshape(-1)
        )

        allele_features = csr_matrix(
            (
                np.ones(
                    len(feature_rows),
                    dtype=allele_count_dtype,
                ),
                (
                    feature_rows,
                    feature_columns,
                ),
            ),
            shape=(
                unique_alleles,
                5 * aligned_positions,
            ),
            dtype=allele_count_dtype,
        )

        allele_matches = np.empty(
            (
                unique_alleles,
                unique_alleles,
            ),
            dtype=allele_count_dtype,
        )

        for allele_start in range(
            0,
            unique_alleles,
            ALLELE_BLOCK_SIZE,
        ):
            allele_stop = min(
                allele_start + ALLELE_BLOCK_SIZE,
                unique_alleles,
            )

            allele_matches[
                allele_start:allele_stop
            ] = (
                allele_features[
                    allele_start:allele_stop
                ]
                @ allele_features.T
            ).toarray()

        for sequence_start in range(
            0,
            len(pathogen_records),
            KERNEL_BLOCK_SIZE,
        ):
            sequence_stop = min(
                sequence_start
                + KERNEL_BLOCK_SIZE,
                len(pathogen_records),
            )

            block_rows = pathogen_row_indices[
                sequence_start:sequence_stop
            ]

            block_values = allele_matches[
                inverse_alleles[
                    sequence_start:sequence_stop
                ][:, None],
                inverse_alleles[None, :],
            ].astype(
                np.uint32,
                copy=False,
            )

            current_values = raw_match_kernel[
                np.ix_(
                    block_rows,
                    pathogen_row_indices,
                )
            ]

            raw_match_kernel[
                np.ix_(
                    block_rows,
                    pathogen_row_indices,
                )
            ] = (
                current_values
                + block_values
            )

        valid_aligned_position_counts[
            pathogen_row_indices
        ] += aligned_positions

        raw_match_kernel.flush()

        print(
            f"[{locus_number}/{len(aligned_loci)}] "
            f"{locus.locus_id}: "
            f"{len(pathogen_records):,} pathogens, "
            f"{unique_alleles:,} distinct aligned sequences"
        )

        del (
            encoded_states,
            unique_states,
            inverse_alleles,
            allele_features,
            allele_matches,
            feature_rows,
            feature_columns,
        )

    if (
        valid_aligned_position_counts
        > np.iinfo(np.uint32).max
    ).any():
        raise OverflowError(
            "The valid-position count exceeds uint32."
        )

    sequence_kernel = np.lib.format.open_memmap(
        LOCAL_KERNEL_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(
            EXPECTED_MODEL_C_PATHOGENS,
            EXPECTED_MODEL_C_PATHOGENS,
        ),
    )

    denominator_vector = np.sqrt(
        valid_aligned_position_counts.astype(
            np.float64
        )
    )

    for row_start in range(
        0,
        EXPECTED_MODEL_C_PATHOGENS,
        KERNEL_BLOCK_SIZE,
    ):
        row_stop = min(
            row_start + KERNEL_BLOCK_SIZE,
            EXPECTED_MODEL_C_PATHOGENS,
        )

        denominator = (
            denominator_vector[
                row_start:row_stop,
                None,
            ]
            * denominator_vector[
                None,
                :,
            ]
        )

        raw_block = np.asarray(
            raw_match_kernel[
                row_start:row_stop
            ],
            dtype=np.float64,
        )

        normalised_block = np.divide(
            raw_block,
            denominator,
            out=np.zeros_like(
                raw_block,
                dtype=np.float64,
            ),
            where=denominator > 0,
        )

        sequence_kernel[
            row_start:row_stop
        ] = normalised_block.astype(
            np.float32
        )

        sequence_kernel.flush()

    del raw_match_kernel
    del sequence_kernel

    temporary_kernel_path = (
        SEQUENCE_KERNEL_PATH.with_suffix(
            ".npy.partial"
        )
    )

    temporary_kernel_path.unlink(
        missing_ok=True
    )

    shutil.copy2(
        LOCAL_KERNEL_PATH,
        temporary_kernel_path,
    )

    temporary_kernel_path.replace(
        SEQUENCE_KERNEL_PATH
    )

    np.save(
        VALID_POSITION_COUNT_PATH,
        valid_aligned_position_counts,
    )

    kernel_metadata = {
        "kernel_name": "K_seq",
        "kernel_shape": [
            EXPECTED_MODEL_C_PATHOGENS,
            EXPECTED_MODEL_C_PATHOGENS,
        ],
        "kernel_dtype": "float32",
        "nucleotide_states": list(
            NUCLEOTIDE_STATES
        ),
        "total_aligned_positions":
            TOTAL_ALIGNED_POSITIONS,
        "total_five_state_dimensions":
            TOTAL_FIVE_STATE_DIMENSIONS,
        "normalisation": (
            "matching aligned states divided by the "
            "geometric mean of the two pathogen-specific "
            "valid-position counts"
        ),
        "missing_sequence_rule": (
            "missing or unreliable locus results contribute "
            "zero and are not encoded as alignment gaps"
        ),
        "alignment_signature":
            alignment_signature,
    }

    temporary_metadata_path = (
        KERNEL_METADATA_PATH.with_suffix(
            ".json.partial"
        )
    )

    with open(
        temporary_metadata_path,
        "w",
        encoding="utf-8",
    ) as metadata_file:
        json.dump(
            kernel_metadata,
            metadata_file,
            indent=2,
        )

    temporary_metadata_path.replace(
        KERNEL_METADATA_PATH
    )

    RAW_KERNEL_WORK_PATH.unlink(
        missing_ok=True
    )
    LOCAL_KERNEL_PATH.unlink(
        missing_ok=True
    )

print(f"Saved: {SEQUENCE_KERNEL_PATH}")
print(f"Saved: {VALID_POSITION_COUNT_PATH}")
print(f"Saved: {KERNEL_METADATA_PATH}")

print(
    "\nTransition: Cell 24.11 will validate the "
    "sequence-kernel dimensions, symmetry, diagonal and range."
)



[1/265] LOCUS_0001: 2 pathogens, 1 distinct aligned sequences
[2/265] LOCUS_0003: 1 pathogens, 1 distinct aligned sequences
[3/265] LOCUS_0004: 2 pathogens, 2 distinct aligned sequences
[4/265] LOCUS_0005: 427 pathogens, 4 distinct aligned sequences
[5/265] LOCUS_0007: 266 pathogens, 5 distinct aligned sequences
[6/265] LOCUS_0008: 2 pathogens, 1 distinct aligned sequences
[7/265] LOCUS_0009: 123 pathogens, 4 distinct aligned sequences
[8/265] LOCUS_0011: 598 pathogens, 33 distinct aligned sequences
[9/265] LOCUS_0012: 2 pathogens, 1 distinct aligned sequences
[10/265] LOCUS_0013: 3 pathogens, 1 distinct aligned sequences
[11/265] LOCUS_0014: 3 pathogens, 1 distinct aligned sequences
[12/265] LOCUS_0015: 18 pathogens, 17 distinct aligned sequences
[13/265] LOCUS_0016: 342 pathogens, 2 distinct aligned sequences
[14/265] LOCUS_0018: 12 pathogens, 1 distinct aligned sequences
[15/265] LOCUS_0019: 6 pathogens, 2 distinct aligned sequences
[16/265] LOCUS_0020: 1 pathogens, 1 distinct align

In [11]:
# =============================================================================
# Cell 24.10B
# =============================================================================

#@title Cell 24.10B - Calculate the locus-weighted sequence kernel
# This cell calculates nucleotide similarity separately within each represented
# locus and gives every available locus equal weight. Missing or unreliable
# locus results contribute zero. The final normalisation preserves a diagonal
# value of 1 for every pathogen with at least one available locus.

from scipy.sparse import csr_matrix

LOCUS_WEIGHTED_RAW_WORK_PATH = (
    WORK_DIRECTORY
    / "24_raw_locus_weighted_similarity_sums.dat"
)

LOCUS_WEIGHTED_LOCAL_KERNEL_PATH = (
    WORK_DIRECTORY
    / "24_sequence_kernel_locus_weighted.npy"
)

LOCUS_WEIGHTED_SEQUENCE_KERNEL_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_sequence_kernel_locus_weighted.npy"
)

AVAILABLE_LOCUS_COUNT_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_available_locus_counts.npy"
)

LOCUS_WEIGHTED_KERNEL_METADATA_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_sequence_kernel_locus_weighted_metadata.json"
)

LOCUS_WEIGHTED_NORMALISATION_NAME = (
    "sum of within-locus matching proportions divided by "
    "the geometric mean of the two pathogen-specific "
    "available-locus counts"
)

reuse_existing_locus_weighted_kernel = False

if (
    LOCUS_WEIGHTED_SEQUENCE_KERNEL_PATH.exists()
    and AVAILABLE_LOCUS_COUNT_PATH.exists()
    and LOCUS_WEIGHTED_KERNEL_METADATA_PATH.exists()
):
    with open(
        LOCUS_WEIGHTED_KERNEL_METADATA_PATH,
        "r",
        encoding="utf-8",
    ) as metadata_file:
        existing_locus_metadata = json.load(
            metadata_file
        )

    reuse_existing_locus_weighted_kernel = (
        existing_locus_metadata.get(
            "alignment_signature"
        )
        == alignment_signature
        and existing_locus_metadata.get(
            "kernel_shape"
        )
        == [
            EXPECTED_MODEL_C_PATHOGENS,
            EXPECTED_MODEL_C_PATHOGENS,
        ]
        and existing_locus_metadata.get(
            "normalisation"
        )
        == LOCUS_WEIGHTED_NORMALISATION_NAME
    )

if reuse_existing_locus_weighted_kernel:
    available_locus_counts = np.load(
        AVAILABLE_LOCUS_COUNT_PATH
    )

    print(
        "The existing locus-weighted sequence kernel matches "
        "the current validated alignments and will be reused."
    )

else:
    LOCUS_WEIGHTED_RAW_WORK_PATH.unlink(
        missing_ok=True
    )
    LOCUS_WEIGHTED_LOCAL_KERNEL_PATH.unlink(
        missing_ok=True
    )

    raw_locus_similarity_kernel = np.memmap(
        LOCUS_WEIGHTED_RAW_WORK_PATH,
        dtype=np.float32,
        mode="w+",
        shape=(
            EXPECTED_MODEL_C_PATHOGENS,
            EXPECTED_MODEL_C_PATHOGENS,
        ),
    )

    raw_locus_similarity_kernel[:] = 0.0

    available_locus_counts = np.zeros(
        EXPECTED_MODEL_C_PATHOGENS,
        dtype=np.uint16,
    )

    locus_state_lookup = np.full(
        256,
        255,
        dtype=np.uint8,
    )

    for state_number, state in enumerate(
        NUCLEOTIDE_STATES
    ):
        locus_state_lookup[ord(state)] = state_number

    represented_loci = alignment_validation[
        alignment_validation[
            "validation_status"
        ]
        == "passed"
    ]

    for locus_number, locus in enumerate(
        represented_loci.itertuples(
            index=False
        ),
        start=1,
    ):
        checkpoint_path = (
            ALIGNMENT_CHECKPOINT_DIRECTORY
            / f"24_{locus.locus_id}_alignment.fasta.gz"
        )

        records = load_gzip_alignment(
            checkpoint_path
        )

        pathogen_records = [
            record
            for record in records
            if record.id != "ANCHOR"
        ]

        pathogen_row_indices = np.array(
            [
                int(record.id[1:])
                for record in pathogen_records
            ],
            dtype=np.int32,
        )

        aligned_length_bp = int(
            locus.aligned_positions
        )

        sequence_bytes = "".join(
            str(record.seq).upper()
            for record in pathogen_records
        ).encode("ascii")

        encoded_states = locus_state_lookup[
            np.frombuffer(
                sequence_bytes,
                dtype=np.uint8,
            )
        ].reshape(
            len(pathogen_records),
            aligned_length_bp,
        )

        if (
            encoded_states == 255
        ).any():
            raise ValueError(
                f"Unexpected state in {locus.locus_id}."
            )

        unique_states, inverse_alleles = np.unique(
            encoded_states,
            axis=0,
            return_inverse=True,
        )

        unique_alleles = len(
            unique_states
        )

        if aligned_length_bp > np.iinfo(
            np.uint16
        ).max:
            allele_count_dtype = np.uint32
        else:
            allele_count_dtype = np.uint16

        feature_rows = np.repeat(
            np.arange(
                unique_alleles,
                dtype=np.int32,
            ),
            aligned_length_bp,
        )

        feature_columns = (
            np.tile(
                np.arange(
                    aligned_length_bp,
                    dtype=np.int32,
                ),
                unique_alleles,
            )
            * 5
            + unique_states.reshape(-1)
        )

        allele_features = csr_matrix(
            (
                np.ones(
                    len(feature_rows),
                    dtype=allele_count_dtype,
                ),
                (
                    feature_rows,
                    feature_columns,
                ),
            ),
            shape=(
                unique_alleles,
                5 * aligned_length_bp,
            ),
            dtype=allele_count_dtype,
        )

        allele_matches = np.empty(
            (
                unique_alleles,
                unique_alleles,
            ),
            dtype=allele_count_dtype,
        )

        for allele_start in range(
            0,
            unique_alleles,
            ALLELE_BLOCK_SIZE,
        ):
            allele_stop = min(
                allele_start + ALLELE_BLOCK_SIZE,
                unique_alleles,
            )

            allele_matches[
                allele_start:allele_stop
            ] = (
                allele_features[
                    allele_start:allele_stop
                ]
                @ allele_features.T
            ).toarray()

        allele_similarities = (
            allele_matches.astype(
                np.float32
            )
            / np.float32(aligned_length_bp)
        )

        for sequence_start in range(
            0,
            len(pathogen_records),
            KERNEL_BLOCK_SIZE,
        ):
            sequence_stop = min(
                sequence_start
                + KERNEL_BLOCK_SIZE,
                len(pathogen_records),
            )

            block_rows = pathogen_row_indices[
                sequence_start:sequence_stop
            ]

            block_values = allele_similarities[
                inverse_alleles[
                    sequence_start:sequence_stop
                ][:, None],
                inverse_alleles[None, :],
            ]

            current_values = raw_locus_similarity_kernel[
                np.ix_(
                    block_rows,
                    pathogen_row_indices,
                )
            ]

            raw_locus_similarity_kernel[
                np.ix_(
                    block_rows,
                    pathogen_row_indices,
                )
            ] = (
                current_values
                + block_values
            )

        available_locus_counts[
            pathogen_row_indices
        ] += 1

        raw_locus_similarity_kernel.flush()

        print(
            f"[{locus_number}/{len(represented_loci)}] "
            f"{locus.locus_id}: "
            f"{len(pathogen_records):,} pathogens, "
            f"{unique_alleles:,} distinct aligned sequences"
        )

        del (
            encoded_states,
            unique_states,
            inverse_alleles,
            allele_features,
            allele_matches,
            allele_similarities,
            feature_rows,
            feature_columns,
        )

    locus_weighted_kernel = np.lib.format.open_memmap(
        LOCUS_WEIGHTED_LOCAL_KERNEL_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(
            EXPECTED_MODEL_C_PATHOGENS,
            EXPECTED_MODEL_C_PATHOGENS,
        ),
    )

    locus_denominator_vector = np.sqrt(
        available_locus_counts.astype(
            np.float64
        )
    )

    for row_start in range(
        0,
        EXPECTED_MODEL_C_PATHOGENS,
        KERNEL_BLOCK_SIZE,
    ):
        row_stop = min(
            row_start + KERNEL_BLOCK_SIZE,
            EXPECTED_MODEL_C_PATHOGENS,
        )

        denominator = (
            locus_denominator_vector[
                row_start:row_stop,
                None,
            ]
            * locus_denominator_vector[
                None,
                :,
            ]
        )

        raw_block = np.asarray(
            raw_locus_similarity_kernel[
                row_start:row_stop
            ],
            dtype=np.float64,
        )

        normalised_block = np.divide(
            raw_block,
            denominator,
            out=np.zeros_like(
                raw_block,
                dtype=np.float64,
            ),
            where=denominator > 0,
        )

        locus_weighted_kernel[
            row_start:row_stop
        ] = normalised_block.astype(
            np.float32
        )

        locus_weighted_kernel.flush()

    del raw_locus_similarity_kernel
    del locus_weighted_kernel

    temporary_kernel_path = (
        LOCUS_WEIGHTED_SEQUENCE_KERNEL_PATH.with_suffix(
            ".npy.partial"
        )
    )

    temporary_kernel_path.unlink(
        missing_ok=True
    )

    shutil.copy2(
        LOCUS_WEIGHTED_LOCAL_KERNEL_PATH,
        temporary_kernel_path,
    )

    temporary_kernel_path.replace(
        LOCUS_WEIGHTED_SEQUENCE_KERNEL_PATH
    )

    np.save(
        AVAILABLE_LOCUS_COUNT_PATH,
        available_locus_counts,
    )

    locus_weighted_metadata = {
        "kernel_name": "K_seq_locus_weighted",
        "kernel_shape": [
            EXPECTED_MODEL_C_PATHOGENS,
            EXPECTED_MODEL_C_PATHOGENS,
        ],
        "kernel_dtype": "float32",
        "represented_loci": int(
            len(represented_loci)
        ),
        "nucleotide_states": list(
            NUCLEOTIDE_STATES
        ),
        "within_locus_similarity": (
            "matching aligned states divided by the "
            "aligned sequence length of that locus"
        ),
        "normalisation": (
            LOCUS_WEIGHTED_NORMALISATION_NAME
        ),
        "missing_sequence_rule": (
            "missing or unreliable locus results contribute "
            "zero and are not encoded as alignment gaps"
        ),
        "zero_available_locus_rule": (
            "similarity is zero if either pathogen has no "
            "available locus sequence"
        ),
        "alignment_signature": alignment_signature,
    }

    temporary_metadata_path = (
        LOCUS_WEIGHTED_KERNEL_METADATA_PATH.with_suffix(
            ".json.partial"
        )
    )

    with open(
        temporary_metadata_path,
        "w",
        encoding="utf-8",
    ) as metadata_file:
        json.dump(
            locus_weighted_metadata,
            metadata_file,
            indent=2,
        )

    temporary_metadata_path.replace(
        LOCUS_WEIGHTED_KERNEL_METADATA_PATH
    )

    LOCUS_WEIGHTED_RAW_WORK_PATH.unlink(
        missing_ok=True
    )
    LOCUS_WEIGHTED_LOCAL_KERNEL_PATH.unlink(
        missing_ok=True
    )

print(
    f"Saved: {LOCUS_WEIGHTED_SEQUENCE_KERNEL_PATH}"
)
print(f"Saved: {AVAILABLE_LOCUS_COUNT_PATH}")
print(
    f"Saved: {LOCUS_WEIGHTED_KERNEL_METADATA_PATH}"
)

print(
    "\nTransition: Cell 24.11 will validate and compare "
    "the base-weighted and locus-weighted sequence kernels."
)

[1/265] LOCUS_0001: 2 pathogens, 1 distinct aligned sequences
[2/265] LOCUS_0003: 1 pathogens, 1 distinct aligned sequences
[3/265] LOCUS_0004: 2 pathogens, 2 distinct aligned sequences
[4/265] LOCUS_0005: 427 pathogens, 4 distinct aligned sequences
[5/265] LOCUS_0007: 266 pathogens, 5 distinct aligned sequences
[6/265] LOCUS_0008: 2 pathogens, 1 distinct aligned sequences
[7/265] LOCUS_0009: 123 pathogens, 4 distinct aligned sequences
[8/265] LOCUS_0011: 598 pathogens, 33 distinct aligned sequences
[9/265] LOCUS_0012: 2 pathogens, 1 distinct aligned sequences
[10/265] LOCUS_0013: 3 pathogens, 1 distinct aligned sequences
[11/265] LOCUS_0014: 3 pathogens, 1 distinct aligned sequences
[12/265] LOCUS_0015: 18 pathogens, 17 distinct aligned sequences
[13/265] LOCUS_0016: 342 pathogens, 2 distinct aligned sequences
[14/265] LOCUS_0018: 12 pathogens, 1 distinct aligned sequences
[15/265] LOCUS_0019: 6 pathogens, 2 distinct aligned sequences
[16/265] LOCUS_0020: 1 pathogens, 1 distinct align

In [12]:
# =============================================================================
# Cell 24.11
# =============================================================================

#@title Cell 24.11 - Validate and compare both sequence kernels
# This cell validates the base-weighted and locus-weighted sequence kernels in
# memory-safe row blocks. It confirms their dimensions, finite values,
# symmetry, diagonal, numerical range and sampled positive-semidefinite
# behaviour. It then compares the two complete matrices.

base_weighted_kernel = np.load(
    SEQUENCE_KERNEL_PATH,
    mmap_mode="r",
)

locus_weighted_kernel = np.load(
    LOCUS_WEIGHTED_SEQUENCE_KERNEL_PATH,
    mmap_mode="r",
)

valid_aligned_length_counts = np.load(
    VALID_POSITION_COUNT_PATH
)

available_locus_counts = np.load(
    AVAILABLE_LOCUS_COUNT_PATH
)

expected_kernel_shape = (
    EXPECTED_MODEL_C_PATHOGENS,
    EXPECTED_MODEL_C_PATHOGENS,
)

expected_support_shape = (
    EXPECTED_MODEL_C_PATHOGENS,
)


def validate_sequence_kernel(
    kernel,
    support_counts,
    kernel_label,
):
    if kernel.shape != expected_kernel_shape:
        raise ValueError(
            f"Expected {kernel_label} shape "
            f"{expected_kernel_shape}, but observed "
            f"{kernel.shape}."
        )

    if support_counts.shape != expected_support_shape:
        raise ValueError(
            f"The {kernel_label} support-count vector "
            "has the wrong dimension."
        )

    kernel_minimum = np.inf
    kernel_maximum = -np.inf
    all_finite = True
    maximum_symmetry_difference = 0.0

    for row_start in range(
        0,
        EXPECTED_MODEL_C_PATHOGENS,
        256,
    ):
        row_stop = min(
            row_start + 256,
            EXPECTED_MODEL_C_PATHOGENS,
        )

        row_block = np.asarray(
            kernel[
                row_start:row_stop,
                :,
            ]
        )

        column_block = np.asarray(
            kernel[
                :,
                row_start:row_stop,
            ]
        ).T

        all_finite = (
            all_finite
            and np.isfinite(row_block).all()
        )

        kernel_minimum = min(
            kernel_minimum,
            float(row_block.min()),
        )

        kernel_maximum = max(
            kernel_maximum,
            float(row_block.max()),
        )

        maximum_symmetry_difference = max(
            maximum_symmetry_difference,
            float(
                np.max(
                    np.abs(
                        row_block
                        - column_block
                    )
                )
            ),
        )

    kernel_diagonal = np.asarray(
        np.diagonal(kernel),
        dtype=np.float64,
    )

    expected_diagonal = np.where(
        support_counts > 0,
        1.0,
        0.0,
    )

    maximum_diagonal_difference = float(
        np.max(
            np.abs(
                kernel_diagonal
                - expected_diagonal
            )
        )
    )

    sample_size = min(
        512,
        EXPECTED_MODEL_C_PATHOGENS,
    )

    sample_indices = np.linspace(
        0,
        EXPECTED_MODEL_C_PATHOGENS - 1,
        sample_size,
        dtype=int,
    )

    sample_kernel = np.asarray(
        kernel[
            np.ix_(
                sample_indices,
                sample_indices,
            )
        ],
        dtype=np.float64,
    )

    sample_minimum_eigenvalue = float(
        np.linalg.eigvalsh(
            sample_kernel
        ).min()
    )

    validation_checks = {
        "dimensions_correct":
            kernel.shape
            == expected_kernel_shape,
        "support_dimension_correct":
            support_counts.shape
            == expected_support_shape,
        "all_values_finite":
            all_finite,
        "symmetric":
            maximum_symmetry_difference
            <= 1e-6,
        "diagonal_correct":
            maximum_diagonal_difference
            <= 1e-6,
        "values_in_zero_one_range":
            kernel_minimum >= -1e-6
            and kernel_maximum <= 1.0 + 1e-6,
        "sample_principal_matrix_psd":
            sample_minimum_eigenvalue
            >= -1e-4,
    }

    if not all(validation_checks.values()):
        failed_checks = [
            check_name
            for check_name, passed
            in validation_checks.items()
            if not passed
        ]

        raise ValueError(
            f"{kernel_label} validation failed: "
            f"{failed_checks}"
        )

    return {
        "kernel": kernel_label,
        "rows": kernel.shape[0],
        "columns": kernel.shape[1],
        "all_values_finite": all_finite,
        "minimum_value": kernel_minimum,
        "maximum_value": kernel_maximum,
        "maximum_symmetry_difference":
            maximum_symmetry_difference,
        "maximum_diagonal_difference":
            maximum_diagonal_difference,
        "sample_minimum_eigenvalue":
            sample_minimum_eigenvalue,
        "pathogens_with_positive_support": int(
            (support_counts > 0).sum()
        ),
        "minimum_support": int(
            support_counts.min()
        ),
        "maximum_support": int(
            support_counts.max()
        ),
        "validation_status": "passed",
    }


base_validation = validate_sequence_kernel(
    base_weighted_kernel,
    valid_aligned_length_counts,
    "Base-weighted sequence kernel",
)

locus_validation = validate_sequence_kernel(
    locus_weighted_kernel,
    available_locus_counts,
    "Locus-weighted sequence kernel",
)

comparison_count = 0
base_sum = 0.0
locus_sum = 0.0
base_square_sum = 0.0
locus_square_sum = 0.0
cross_product_sum = 0.0
absolute_difference_sum = 0.0
squared_difference_sum = 0.0
maximum_absolute_difference = 0.0

for row_start in range(
    0,
    EXPECTED_MODEL_C_PATHOGENS,
    256,
):
    row_stop = min(
        row_start + 256,
        EXPECTED_MODEL_C_PATHOGENS,
    )

    base_block = np.asarray(
        base_weighted_kernel[
            row_start:row_stop,
            :,
        ],
        dtype=np.float64,
    )

    locus_block = np.asarray(
        locus_weighted_kernel[
            row_start:row_stop,
            :,
        ],
        dtype=np.float64,
    )

    difference_block = (
        base_block - locus_block
    )

    comparison_count += base_block.size
    base_sum += float(base_block.sum())
    locus_sum += float(locus_block.sum())
    base_square_sum += float(
        np.square(base_block).sum()
    )
    locus_square_sum += float(
        np.square(locus_block).sum()
    )
    cross_product_sum += float(
        (base_block * locus_block).sum()
    )
    absolute_difference_sum += float(
        np.abs(difference_block).sum()
    )
    squared_difference_sum += float(
        np.square(difference_block).sum()
    )
    maximum_absolute_difference = max(
        maximum_absolute_difference,
        float(
            np.abs(difference_block).max()
        ),
    )

correlation_numerator = (
    comparison_count * cross_product_sum
    - base_sum * locus_sum
)

correlation_denominator = np.sqrt(
    (
        comparison_count * base_square_sum
        - base_sum ** 2
    )
    * (
        comparison_count * locus_square_sum
        - locus_sum ** 2
    )
)

if correlation_denominator > 0:
    full_matrix_correlation = float(
        correlation_numerator
        / correlation_denominator
    )
else:
    full_matrix_correlation = np.nan

mean_absolute_difference = float(
    absolute_difference_sum
    / comparison_count
)

root_mean_squared_difference = float(
    np.sqrt(
        squared_difference_sum
        / comparison_count
    )
)

KERNEL_VALIDATION_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_sequence_kernel_candidate_validation.csv"
)

KERNEL_COMPARISON_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_sequence_kernel_candidate_comparison.csv"
)

PATHOGEN_ORDER_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_model_c_pathogen_order.csv"
)

SEQUENCE_COVERAGE_TABLE_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_pathogen_sequence_coverage.csv"
)

kernel_validation = pd.DataFrame(
    [
        base_validation,
        locus_validation,
    ]
)

kernel_comparison = pd.DataFrame(
    [
        {
            "metric": "Compared matrix cells",
            "value": comparison_count,
        },
        {
            "metric": "Full-matrix Pearson correlation",
            "value": full_matrix_correlation,
        },
        {
            "metric": "Mean absolute difference",
            "value": mean_absolute_difference,
        },
        {
            "metric": "Root mean squared difference",
            "value": root_mean_squared_difference,
        },
        {
            "metric": "Maximum absolute difference",
            "value": maximum_absolute_difference,
        },
        {
            "metric": "Selection status",
            "value": (
                "Both kernels retained for predictive "
                "comparison in Notebook 25"
            ),
        },
    ]
)

kernel_validation.to_csv(
    KERNEL_VALIDATION_PATH,
    index=False,
)

kernel_comparison.to_csv(
    KERNEL_COMPARISON_PATH,
    index=False,
)

pathogen_order = progress[
    [
        "model_c_row_index",
        "model_3b_row_index",
        "biosample",
        "assembly_accession",
    ]
].copy()

pathogen_order.to_csv(
    PATHOGEN_ORDER_PATH,
    index=False,
)

sequence_coverage_table = pathogen_order[
    [
        "model_c_row_index",
        "biosample",
        "assembly_accession",
    ]
].copy()

sequence_coverage_table[
    "total_aligned_sequence_length_bp"
] = valid_aligned_length_counts

sequence_coverage_table[
    "available_loci"
] = available_locus_counts

sequence_coverage_table.to_csv(
    SEQUENCE_COVERAGE_TABLE_PATH,
    index=False,
)

display(kernel_validation)
display(kernel_comparison)

print(f"Saved: {KERNEL_VALIDATION_PATH}")
print(f"Saved: {KERNEL_COMPARISON_PATH}")
print(f"Saved: {PATHOGEN_ORDER_PATH}")
print(f"Saved: {SEQUENCE_COVERAGE_TABLE_PATH}")

print(
    "\nTransition: Cell 24.12 will package both validated "
    "sequence-kernel candidates and report the final status."
)


,kernel,rows,columns,all_values_finite,minimum_value,maximum_value,maximum_symmetry_difference,maximum_diagonal_difference,sample_minimum_eigenvalue,pathogens_with_positive_support,minimum_support,maximum_support,validation_status
0,Base-weighted sequence kernel,9058,9058,True,0.675245,1.0,0.0,0.0,-1.591075e-08,9058,39052,67636,passed
1,Locus-weighted sequence kernel,9058,9058,True,0.586458,1.0,0.0,0.0,-1.125484e-08,9058,30,59,passed


,metric,value
0,Compared matrix cells,82047364
1,Full-matrix Pearson correlation,0.974056
2,Mean absolute difference,0.029546
3,Root mean squared difference,0.035044
4,Maximum absolute difference,0.130941
5,Selection status,Both kernels retained for predictive compariso...


Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_sequence_kernel_candidate_validation.csv
Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_sequence_kernel_candidate_comparison.csv
Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_model_c_pathogen_order.csv
Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/results/24_pathogen_sequence_coverage.csv

Transition: Cell 24.12 will package both validated sequence-kernel candidates and report the final status.


In [13]:
# =============================================================================
# Cell 24.12
# =============================================================================

#@title Cell 24.12 - Package and report the final Notebook 24 outputs
# This cell records checksums, packages both validated sequence-kernel
# candidates and their supporting files into one ZIP archive, and reports
# whether Notebook 24 completed successfully.

FINAL_OUTPUT_ARCHIVE_PATH = (
    NOTEBOOK24_DIRECTORY
    / "24_targeted_amr_sequence_kernel_candidates_outputs.zip"
)

OUTPUT_MANIFEST_PATH = (
    NOTEBOOK24_RESULT_DIRECTORY
    / "24_output_manifest.json"
)

files_to_package = [
    SEQUENCE_KERNEL_PATH,
    VALID_POSITION_COUNT_PATH,
    KERNEL_METADATA_PATH,
    LOCUS_WEIGHTED_SEQUENCE_KERNEL_PATH,
    AVAILABLE_LOCUS_COUNT_PATH,
    LOCUS_WEIGHTED_KERNEL_METADATA_PATH,
    KERNEL_VALIDATION_PATH,
    KERNEL_COMPARISON_PATH,
    PATHOGEN_ORDER_PATH,
    SEQUENCE_COVERAGE_TABLE_PATH,
    LOCUS_AVAILABILITY_PATH,
    SELECTED_SEQUENCE_PATH,
    ALIGNMENT_ANCHOR_PATH,
    ALIGNMENT_ANCHOR_FASTA_PATH,
    ALIGNMENT_VALIDATION_PATH,
    ARCHIVE_AUDIT_PATH,
    CONSOLIDATED_REVIEW_PATH,
]

missing_output_files = [
    path
    for path in files_to_package
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "Notebook 24 output files are missing: "
        f"{missing_output_files}"
    )

if not (
    kernel_validation["validation_status"]
    == "passed"
).all():
    raise ValueError(
        "Both sequence-kernel candidates must pass "
        "validation before packaging."
    )

with open(
    KERNEL_METADATA_PATH,
    "r",
    encoding="utf-8",
) as metadata_file:
    base_weighted_metadata = json.load(
        metadata_file
    )

with open(
    LOCUS_WEIGHTED_KERNEL_METADATA_PATH,
    "r",
    encoding="utf-8",
) as metadata_file:
    locus_weighted_metadata = json.load(
        metadata_file
    )

output_manifest = {
    "notebook": 24,
    "model": "Model C",
    "model_c_pathogens":
        EXPECTED_MODEL_C_PATHOGENS,
    "planned_targeted_loci":
        EXPECTED_LOCI,
    "represented_loci": int(
        (
            alignment_validation[
                "validation_status"
            ]
            == "passed"
        ).sum()
    ),
    "total_aligned_sequence_length_bp":
        TOTAL_ALIGNED_POSITIONS,
    "total_five_state_dimensions":
        TOTAL_FIVE_STATE_DIMENSIONS,
    "kernel_shape": [
        EXPECTED_MODEL_C_PATHOGENS,
        EXPECTED_MODEL_C_PATHOGENS,
    ],
    "kernel_dtype": "float32",
    "kernel_candidates": [
        {
            "candidate_name":
                "base-weighted sequence kernel",
            "kernel_file":
                SEQUENCE_KERNEL_PATH.name,
            "support_file":
                VALID_POSITION_COUNT_PATH.name,
            "metadata_file":
                KERNEL_METADATA_PATH.name,
            "normalisation":
                base_weighted_metadata.get(
                    "normalisation"
                ),
            "validation_status": "passed",
        },
        {
            "candidate_name":
                "locus-weighted sequence kernel",
            "kernel_file":
                LOCUS_WEIGHTED_SEQUENCE_KERNEL_PATH.name,
            "support_file":
                AVAILABLE_LOCUS_COUNT_PATH.name,
            "metadata_file":
                LOCUS_WEIGHTED_KERNEL_METADATA_PATH.name,
            "normalisation":
                locus_weighted_metadata.get(
                    "normalisation"
                ),
            "validation_status": "passed",
        },
    ],
    "kernel_selection_status": (
        "Both candidates retained for predictive "
        "comparison in Notebook 25"
    ),
    "files": [
        {
            "file_name": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        }
        for path in files_to_package
    ],
}

temporary_manifest_path = (
    OUTPUT_MANIFEST_PATH.with_suffix(
        ".json.partial"
    )
)

with open(
    temporary_manifest_path,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        output_manifest,
        manifest_file,
        indent=2,
    )

temporary_manifest_path.replace(
    OUTPUT_MANIFEST_PATH
)

files_to_package.append(
    OUTPUT_MANIFEST_PATH
)

local_archive_path = (
    WORK_DIRECTORY
    / FINAL_OUTPUT_ARCHIVE_PATH.name
)

local_archive_path.unlink(
    missing_ok=True
)

with zipfile.ZipFile(
    local_archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True,
) as archive:
    for file_path in files_to_package:
        archive.write(
            file_path,
            arcname=file_path.name,
        )

with zipfile.ZipFile(
    local_archive_path,
    "r",
) as archive:
    damaged_member = archive.testzip()

    if damaged_member is not None:
        raise ValueError(
            "The final archive contains a damaged file: "
            f"{damaged_member}"
        )

    archived_members = set(
        archive.namelist()
    )

expected_members = {
    path.name
    for path in files_to_package
}

if archived_members != expected_members:
    raise ValueError(
        "The final archive member list is incomplete."
    )

local_archive_sha256 = file_sha256(
    local_archive_path
)

partial_archive_path = (
    FINAL_OUTPUT_ARCHIVE_PATH.with_suffix(
        ".zip.partial"
    )
)

partial_archive_path.unlink(
    missing_ok=True
)

shutil.copy2(
    local_archive_path,
    partial_archive_path,
)

if (
    file_sha256(partial_archive_path)
    != local_archive_sha256
):
    raise IOError(
        "The copied final archive does not match "
        "the locally validated archive."
    )

partial_archive_path.replace(
    FINAL_OUTPUT_ARCHIVE_PATH
)

with zipfile.ZipFile(
    FINAL_OUTPUT_ARCHIVE_PATH,
    "r",
) as archive:
    if archive.testzip() is not None:
        raise ValueError(
            "The saved final archive failed validation."
        )

final_summary = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "metric": "Planned targeted loci",
            "value": EXPECTED_LOCI,
        },
        {
            "metric": "Represented loci",
            "value": int(
                (
                    alignment_validation[
                        "validation_status"
                    ]
                    == "passed"
                ).sum()
            ),
        },
        {
            "metric": "Total aligned sequence length (bp)",
            "value": TOTAL_ALIGNED_POSITIONS,
        },
        {
            "metric": "Five-state sequence dimensions",
            "value": TOTAL_FIVE_STATE_DIMENSIONS,
        },
        {
            "metric": "Validated sequence-kernel candidates",
            "value": int(
                (
                    kernel_validation[
                        "validation_status"
                    ]
                    == "passed"
                ).sum()
            ),
        },
        {
            "metric": "Each kernel dimension",
            "value": (
                f"{EXPECTED_MODEL_C_PATHOGENS}"
                f" × "
                f"{EXPECTED_MODEL_C_PATHOGENS}"
            ),
        },
        {
            "metric": "Kernel selection status",
            "value": (
                "Deferred to predictive comparison "
                "in Notebook 25"
            ),
        },
        {
            "metric": "Final archive size (GB)",
            "value": round(
                FINAL_OUTPUT_ARCHIVE_PATH.stat().st_size
                / (1024 ** 3),
                3,
            ),
        },
        {
            "metric": "Notebook 24 validation status",
            "value": "passed",
        },
    ]
)

display(final_summary)

print(
    f"Saved: {FINAL_OUTPUT_ARCHIVE_PATH}"
)

print(
    "\nNotebook 24 completed successfully. "
    "Both validated sequence-kernel candidates are "
    "ready for predictive comparison in Notebook 25."
)

,metric,value
0,Model C pathogens,9058
1,Planned targeted loci,297
2,Represented loci,265
3,Total aligned sequence length (bp),247100
4,Five-state sequence dimensions,1235500
5,Validated sequence-kernel candidates,2
6,Each kernel dimension,9058 × 9058
7,Kernel selection status,Deferred to predictive comparison in Notebook 25
8,Final archive size (GB),0.582
9,Notebook 24 validation status,passed


Saved: /content/drive/MyDrive/Model3_MIC_Project/notebook24/24_targeted_amr_sequence_kernel_candidates_outputs.zip

Notebook 24 completed successfully. Both validated sequence-kernel candidates are ready for predictive comparison in Notebook 25.
